# <center>Harness Engineering L5：Curator 整理机制 + 自进化的诚实边界</center>

&emsp;&emsp;欢迎来到 `Harness Engineering` 系列的**最后一节课**。

&emsp;&emsp;前四节课走完了一条完整的主链路：`harness`（工程外壳，类似给 AI 模型装上工具栏和仪表盘）是什么 → 怎么手搓 → 怎么用框架装 → 怎么读生产源码。今天是终章，把最后三块拼图拼完，让你带走一把能评价任何 agent 产品的选型尺。

&emsp;&emsp;今天三条主线，对应三个核心问题：

- **第 1 章**：`Curator`——skill 库越来越多，谁来帮你整理？

- **第 2 章**：血统隔离——用户手写的 skill，会不会被 agent 偷偷覆盖？

- **第 3 章**：诚实边界——"越用越强"这句话，哪里是真的，哪里被夸大了？

&emsp;&emsp;遇到陌生术语不要慌——每次首次出现都会就近解释，也可以随时翻到 §0.5 术语速查表。

&emsp;&emsp;> **📋 课前须知（花 1 分钟读完）**
>
> **需要哪些基础？**
> - 完成了前四节课，能看懂第二节课手搓的 11 个机制清单
> - 记得第四节课的 `Nudge`（计数器触发 skill 写入）和三层记忆架构的基本概念
> - Python 基础 + 能改 YAML/JSON 配置 + 会用命令行
> - **不需要**从零写新框架代码——主线是"读原理 + 跑 Demo + 看源码"
>
> **今天会做什么？**
> - 跑 3 个 Demo：Curator dry-run（第 1 章）/ 血统验证（第 2 章）/ 30 行 GEPA 真跑（第 3 章）
> - 亲眼看到一句极弱的 prompt 被 GEPA 自动进化成一段精确的多条规则文本
> - 用"四维评价尺"横评四家 harness，拿到一套可复用的选型框架
>
> **今天学不到什么（提前说清楚）？**
> - 不会让模型本身变聪明——模型权重不会因为用 Hermes 而改变
> - 不会证明营销材料里"10-20 次后 2-3 倍效率提升"那个数字，没有可复现的 benchmark
> - 不教你训练或微调自己的模型

## <center>第0章：开始之前——建立地图</center>

&emsp;&emsp;第 0 章不讲新知识，只做三件事：先纠正一个常见误区，再建立今天的学习地图，最后把三个 Demo 共用的环境一次性配好。读完第 0 章，进入第 1 章时你心里就有完整的坐标系。

### 0.1 先纠正一个误区：装上 Hermes 就能自动进化吗？

&emsp;&emsp;很多人第一次听说 `Hermes` 时会有一个直觉——**"官方说它'越用越强'，那 skill 自动整理 + 自动进化应该是装上就有的默认功能吧？"**

&emsp;&emsp;答案是：**不全是**，两个功能的实际情况差别很大：

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 功能 | 在哪里 | 能否开箱即用 |
| :---: | :---: | :---: |
| `Curator`（自动整理 skill 库） | 主仓默认链路 | 装上就能跑，按空闲触发、按周扫一次 |
| `GEPA Skill Evolution`（自动进化 skill 内容） | 独立子项目 `hermes-agent-self-evolution` | 需要单独 `git clone` + 配置 `dspy` |

&emsp;&emsp;更重要的是：独立子项目的 `README` 标记了 `Phase 1 [√]`，看起来已经完成。但实际代码和当前 DSPy 的 API 存在明显不匹配，**不能直接按"已实现"来理解**，需要亲手复核。

&emsp;&emsp;> ⚠️ **核心结论**：`README` 上打了勾，不等于端到端可跑。任何 agent 产品的卖点，都需要你亲手跑代码来验证——这是本课第 3 章要教的核心方法论。

&emsp;&emsp;下面这张表列出了今天会反复碰到的五个常见误区和真实情况对比。**现在不需要全部记住**——后面每章讲完都会回来把对应那行勾掉，用来检验自己是否真的理解了。表 0-1 是全课导引图，每章讲完回来勾一行。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-1 常见误区 vs 源码真相（每章讲完回来勾一行）</font></p>
<div class="center">

| 你以为的 | 真实情况 | 哪一节讲清楚 |
|---|---|---|
| `GEPA` 装上 hermes 就能跑，是默认功能 | 独立子项目，需单独安装；且 README 的"已完成"标记需亲手复核 | §3.3 + §3.4 Demo 3 |
| `Curator` 是定时任务（cron），按时间自动跑 | 不是——是"空闲触发"，等你不用 Hermes 的时候才偷偷跑 | §1.2 触发机制 |
| `Curator` 会删掉旧 skill | 永远不删——最大的动作是"归档"（移到隐藏目录），归档是可以恢复的 | §1.3 + §1.5 |
| 用户手写的 skill 也会被 Curator 整理 | 不会——Curator 只动 agent 自己写的 skill，用户手写的有专门的保护机制 | 第 2 章全章 |
| "越用越强" = 模型变聪明了 | 不是——是注入给模型的 prompt 质量在提升，模型本身没变 | §3.2 表 3-1 |

</div>


&emsp;&emsp;打碎这些误区之后，我们已经站在了今天课程的入口——不是问"Hermes 是不是真自进化"，而是问"它的进化机制是什么、真实边界在哪里、哪些卖点可以相信、哪些需要打折"。这正是今天**四维评价尺**要做的事。

&emsp;&emsp;今天本课是这个系列的**终章**——把 `Curator` 整理机制、`ContextVar` 血统隔离、`GEPA` 自进化诚实边界三块拼图收完，再用**四维评价尺**横评四家 harness，拿到一把可复用于任何 agent 产品的评价工具。下面这张图标出本讲在五讲时间轴上的位置：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141448134.png" width=50%></div>

### 0.2 今天的三条主线

&emsp;&emsp;公开课讲义引用厂商话术时，容易让人产生一个错觉——"这个 agent 越用越强，是个会自我进化的智能体"。这个错觉的危险在于把两件完全不同的事混为一谈：**机制层的进化**（prompt 质量提升、skill 库扩充）和**模型层的进化**（参数更新、推理能力变强）。今天三条主线正是要把这层迷雾拨开。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141448702.png" width=60%></div>

**主线一（第 1 章）：skill 库膨胀，谁来质检？**

&emsp;&emsp;用 `Hermes` 久了，agent 每隔一段时间就会自动写一条 skill（经验卡片）落盘。跑一年可能产生几百条 skill，但其中可能一半是重复的、质量差的、过时的。光靠数量增长，skill 库很快会变成"垃圾场"——prompt 注入成本激增，但实际质量下降。`Curator` 就是为了解决这个问题而设计的：它像"图书馆里定期整理书架的管理员"，趁你不用 Hermes 的空档悄悄跑，把 skill 库整理干净。

**主线二（第 2 章）：用户手写的 skill，受保护吗？**

&emsp;&emsp;这是 Reddit 上被顶了 107 票的痛点吐槽。用户辛苦把一条 skill 改对了，agent 下个周期的自动写入直接把用户的修改覆盖回去了。`Hermes` 团队在 5 月的两个修补提交里用 78 行 Python 标准库代码解决了这个问题——通过给每条 skill 打一个隐形的"谁写的"标签（血统标签），`Curator` 后续只处理 agent 自己写的，用户写的完全不碰。

**主线三（第 3 章）："越用越强"，哪里是真的？**

&emsp;&emsp;我们用源码逐条对照厂商的卖点——哪些说得极准、哪些是真但要打折、哪些被夸大了。核心工具是 30 行 `dspy.GEPA` 真跑：你亲眼看到 prompt 从一个字 `"Reply."` 自动进化成一段六条规则的英文说明，accuracy 从 17% 升到 100%。最后用一份**四维评价尺**，让你以后面对任何 agent 产品都能自己拆开看。

<p align="center">表 0-2 第四节课 → 本课 学习路径对比</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 维度 | 第四节课（已学） | 本课（今天） |
| :---: | :---: | :---: |
| 主题动作 | 看 skill **怎么生**（生产端 `Nudge`） | 看 skill **生多了怎么管**（质检端 `Curator`） |
| 你的动作 | 读源码 + 跑 MVP demo | 读源码 + 跑 dry-run + 现场 grep + **亲手跑 GEPA** |
| 核心机制 | `Nudge` 计数器 + fork agent + 落盘 | `Curator` 双阶段 + 血统隔离 + 四维横评 |
| 教学关键词 | 三层记忆 / 19 平台 / 8 Provider | 空闲触发 / 永远不删 / README 打勾不等于跑通 |
| 评价工具 | **四维评价尺**（首次抽出） | **四维四家横评**（系列终章用一次） |


&emsp;&emsp;> **本课新增的核心动作**：第 3 章的 GEPA 演示，你不只是被动看结论，而是**亲手跑 30 行代码**，看 prompt 自动进化的真实形态。这是这个系列最后一节课要建立的核心能力：**看到任何 agent 产品的卖点，第一反应是"我能不能 30 行代码复现它的核心机制"**——这才是工程师看卖点的正确姿势。

### 0.3 业界背景：自进化系统为什么容易翻车

&emsp;&emsp;在打开源码之前，先建立一个心理预期——不是为了打击信心，而是为了进入第 3 章时不被"独立子项目跑不通"这种现象吓到。

&emsp;&emsp;Reddit 上关于 `Hermes` 等自学习 agent 产品的真实反馈里，有一条顶了 107 票的吐槽：

&emsp;&emsp;> *"It always thinks it did a good job. ALWAYS."*
> （它总觉得自己做得很好。永远是这样。）

&emsp;&emsp;这句话精准描述了**自评估过度自信**这个失败模式——让 agent 自己给自己打分，它会系统性地高估。这不是 `Hermes` 一家的问题，所有用"LLM 自己反思"做自学习的系统都有这个缺陷，包括今天我们会跑的 `dspy.GEPA`，它的反思同样是由 LLM 来做的。

&emsp;&emsp;另外，Reddit 用户还反复提到"手动编辑被覆盖""小模型工具调用频繁出错""配置迁移灾难"等问题。其中"手动编辑被覆盖"已经在 5 月被修复——这正是第 2 章血统隔离的来历。但其他问题仍然是系统性挑战，不是用一个版本更新就能全部解决的。

&emsp;&emsp;今天进入第 3 章看 GEPA 实证时，如果发现独立子项目跑不通，不用惊讶——这是行业常态。**我们的目的不是揭穿哪家不诚实，而是学会用 30 行代码亲手验证一个 agent 算法的真实形态**。这种方法论比任何对比表都更硬核。

&emsp;&emsp;> **正确姿势**：批判性思维不是"什么都不信"的怀疑论。`Hermes` 有些设计写得极准——比如 `Frozen Snapshot`（system prompt 状态的冻结记录）的注释能与源码逐字对应；但另一些卖点（比如 GEPA 自进化子项目）需要现场复核。**用 30 行代码亲手跑算法核心**——这才是看清"卖点真假"最直接的路径。

### 0.4 环境准备（三个 Demo 共用，只配一次）

&emsp;&emsp;本课共 3 个 Demo，所有依赖在这里**一次性装好**，后续三章不再重复 `pip install` / `.env` / 加载环境变量这三件套。

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| Demo | 位置 | 依赖什么 |
| :---: | :---: | :---: |
| Demo 1：Curator dry-run | 第 1 章末尾 §1.5.1 | hermes CLI（路径 A）或 DeepSeek API Key（路径 B 二选一） |
| Demo 2：血统验证 | 第 2 章末尾 §2.4.1 | 本地 HermesAgent 源码（可选，教学 stub 版不依赖） |
| Demo 3：30 行 dspy.GEPA | §3.4 | DeepSeek API Key（课件有 cache，不真跑也能看结果） |

**步骤零：创建隔离的 conda 环境**

&emsp;&emsp;为了让你的本地实测结果先创建一个独立的 Python 环境：

In [25]:
!python --version

Python 3.11.15


**步骤一：创建 `.env` 文件**

&emsp;&emsp;在 `ipynb` **同级目录**创建 `.env` 文件，内容如下（把 `your_deepseek_key_here` 替换成你自己的 API Key）：

```dotenv
# DeepSeek API Key（Demo 1 路径 B + Demo 3 需要）
DEEPSEEK_API_KEY=your_deepseek_key_here
DEEPSEEK_MODEL=deepseek-chat

# 本地 Hermes 源码路径（Demo 2 真源码版需要；用教学 stub 版可以留空）
HERMES_REPO_PATH=/Users/your_name/Git/HermesAgent

# Hermes 数据目录（Demo 1 模拟模式可选；真实安装会自动用 ~/.hermes）
HERMES_HOME=~/.hermes
```

&emsp;&emsp;> 🔒 **安全提示**：`.env` 文件包含 API Key 等敏感凭证，**切勿提交到 Git**。请确认 `.gitignore` 里已经有 `.env` 这一行。

**步骤二：一键安装依赖**

&emsp;&emsp;课件目录下提供了 `requirements.txt`，一行命令装齐所有依赖：

In [ ]:
!pip install -r requirements.txt -q

&emsp;&emsp;> 📌 国内网络访问 PyPI 慢？改用清华镜像：`!pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple -q`

&emsp;&emsp;`requirements.txt` 核心内容预览（精确锁版本，避免跨学员环境漂移）：

```text
# ── Jupyter 环境（必需）──
jupyter==1.1.1
notebook==7.2.2
ipykernel==6.29.5

# ── Demo 3：GEPA 反思变异 ──
dspy-ai==3.2.1              # harness env 实测版本；包含 dspy + gepa

# ── Demo 1 路径 B：LangChain 双 Phase 模拟 ──
langchain==1.2.15           # harness env 实测版本
langchain-core==1.3.2
langchain-deepseek==1.0.1   # ChatDeepSeek
langchain-community==0.4.1  # create_agent 等工具函数

# ── 通用工具 ──
python-dotenv==1.0.1        # 读取 .env 中的 API KEY
```

**步骤三：加载 `.env` 并验证**

&emsp;&emsp;下面这段代码是全课**唯一**的环境变量加载 cell，Demo 1/2/3 共用，后面不再重复：

In [1]:
# Demo 1/2/3 共用：加载 .env 并验证关键变量
from dotenv import find_dotenv, load_dotenv
import os

# find_dotenv(usecwd=True) 从当前目录向上查找 .env，兼容 notebook / 终端 / stdin 三种运行方式
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)

# Demo 1 路径 B + Demo 3 必需：DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")
if api_key:
    print(f"[OK] DEEPSEEK_API_KEY 加载成功（{api_key[:6]}...{api_key[-4:]}）")
else:
    print("[INFO] DEEPSEEK_API_KEY 未配置（Demo 1 路径 B 和 Demo 3 需要，Demo 2 不需要）")

# Demo 2 可选：本地 HermesAgent 源码路径
repo_path = os.getenv("HERMES_REPO_PATH")
if repo_path and os.path.isdir(repo_path):
    print(f"[OK] HERMES_REPO_PATH = {repo_path}")
else:
    print("[INFO] HERMES_REPO_PATH 未配置或路径不存在（Demo 2 教学 stub 版不需要）")

# Demo 1 可选：hermes 数据目录
print(f"[INFO] HERMES_HOME = {os.getenv('HERMES_HOME', '~/.hermes（默认）')}")

DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
DEMO3_LM = os.getenv("DEMO3_LM", "deepseek/deepseek-chat")
print(f"[INFO] DEEPSEEK_MODEL = {DEEPSEEK_MODEL}")
print(f"[INFO] DEMO3_LM = {DEMO3_LM}")

[OK] DEEPSEEK_API_KEY 加载成功（sk-d51...014d）
[INFO] HERMES_REPO_PATH 未配置或路径不存在（Demo 2 教学 stub 版不需要）
[INFO] HERMES_HOME = ~/.hermes（默认）
[INFO] DEEPSEEK_MODEL = deepseek-chat
[INFO] DEMO3_LM = deepseek/deepseek-chat


&emsp;&emsp;执行后看到 `[OK]` 表示必填项配置好了；`[INFO]` 是可选项，没有也不影响后续章节的主线流程，遇到对应 Demo 再补填即可。

### 0.5 术语速查表（遇到不认识的词就来这里查）

&emsp;&emsp;本课不打断主线节奏，会直接使用第四节课已经讲过的术语。**不需要现在全部记住**——遇到不认识的词，回来查一下就好。本课新增的术语（`mark_agent_created` / `background_review` / `agent_created_report` 等）不在表中，会在第 2 章首次出现时详细介绍。

<p align="center">表 0-3 第四节课已讲术语速查（遇到再来查，不需要现在背）</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 术语 | 一句话解释 | 本课首现章节 |
| :---: | :---: | :---: |
| `skill_manage(action=...)` | Hermes 内置的 skill 管理工具，支持创建/列出/获取/删除/修改，`create` 时把 skill 写到 `~/.hermes/skills/<name>/SKILL.md` | §1.3 |
| `_spawn_background_review` | Nudge 计数器到了阈值后调用此函数，在后台悄悄 fork 一个独立的 review agent 跑反思，不打断你的主对话 | §2.2 |
| `skill_usage` 模块 | 维护 skill 使用记录的工具模块，记录每条 skill 的最近使用时间、使用次数、创建者、是否被 pin 等字段 | §1.3 |
| `fork review agent` | Hermes 在后台"分叉"出一个独立子 agent——有自己的 LLM 调用、自己的工具集、自己的上下文，跑完即销毁，不影响主对话 | 第 1 章 |
| `SKILL.md` 文件 | 每条 skill 的主体文件，包含 frontmatter（name / description / triggers）+ 正文 markdown，放在 `~/.hermes/skills/<name>/SKILL.md` | 第 1 章 |
| `turn`（对话回合） | agent 的一次完整交互：用户说一句话 → agent 调用若干工具 → 给用户回复，整个算一个 turn；Nudge 计数器按 turn 累加 | §1.1 |

&emsp;&emsp;> 💡 **提示**：某个术语完全没印象时，建议先回看第四节课对应章节，再继续往下读——基础不稳的话，后面的内容会越来越难跟上。只是暂时想不起来用法细节时，扫一眼这张表就好。

&emsp;&emsp;第 0 章结束。环境配好、地图在手，接下来进入第 1 章——打开 `Curator` 的源码，看"扫地僧"究竟是怎么工作的。

## <center>第1章：Curator 定期整理机制</center>

&emsp;&emsp;这一章我们打开 `Hermes` 主仓默认链路里的 `Curator`——口语别名"扫地僧"。它负责把 `Nudge` 自动写出来的 skill，在 `active / stale / archived` 三个状态之间迁移，并把同类的碎片化 skill 合并成一个"大伞 skill"（`umbrella`）。

&emsp;&emsp;第四节课已经看到：`Nudge` 是 skill 库的**生产端**——计数器到了阈值就 fork 一个 agent 写 skill 落盘。但生产端没有质检，跑 1000 次任务可能产出 1000 条 skill，其中一部分重复、一部分质量差、一部分是 agent 过度自信写出的"虚假产出"。`Curator` 就是为了解决这个问题：趁 Hermes 空闲时悄悄跑，扫一遍 skill 库，做好质检。

&emsp;&emsp;`Curator` 的核心动作分两个阶段：**Phase 1 自动状态机**（纯函数，不调 LLM，只看时间戳，把 skill 在三态之间迁移）+ **Phase 2 LLM Review**（fork 一个子 agent，用 LLM 判断哪些 skill 可以合并成 umbrella）。整章按"为什么需要 → 触发机制 → Phase 1 → Phase 2 → 五条护栏 → Demo 1"的顺序逐层展开。

### 1.1 为什么需要 Curator：Nudge 的生产端缺了什么

&emsp;&emsp;**先打碎一个直觉**——很多人第一次看到 `Curator` 这个名字会以为它是 `cron daemon`（按时间表跑的后台服务），或者是个"删过期 skill 的清理工"。**两个都不对**。`Curator` 是 `idle-triggered`（闲置触发），不是定时；它**永远不删**只 archive；它处理的也不是"过期"而是"长期未使用"。先把这三个错误印象打掉，再看它真正做什么。

&emsp;&emsp;回顾一下 `Nudge` 的工作方式：agent 每完成一个 turn，`Nudge counter` 加 1；计数器到了阈值，就 fork 一个 background agent，写一条新 skill 落盘到 `~/.hermes/skills/<name>/SKILL.md`，然后 counter 清零。这个机制让"agent 跑得越久、skill 越丰富"的承诺是真实的。

&emsp;&emsp;但放到生产环境，会立刻暴露三个问题：

&emsp;&emsp;**① 虚假产出**：agent 的 self-evaluation 过度自信，经常觉得"这次做得很完美，值得写一条 skill"，但实际可能只是把同一个常识换了个说法。Reddit 上那条 107 票高赞吐槽说的就是这个——*"It always thinks it did a good job. ALWAYS."*

&emsp;&emsp;**② patch 腐蚀**：`Nudge` 写的 skill 后续会被另一个 agent 反复修改，patch 多了之后，skill 内容越来越长、结构越来越乱，跟最初的意图越来越远。

&emsp;&emsp;**③ token 爆炸**：agent 做任务时每次都要在 prompt 里注入 skill 列表。skill 库膨胀到 200 条以上时，光扫描元数据就吃掉几千 token——如果其中一半是重复的，浪费更严重。

&emsp;&emsp;> **【常见误区】**：看到 `Nudge` 自动写 skill 这个机制，第一反应可能是"agent 自己产 skill 是好事，越多越好"。实际上**未经管理的 skill 库会越用越烂**——这是 `Hermes` 在 5 月之后专门加 `Curator` 的根本原因。判断你自己是否踩了这个坑很简单：跑 30 天后看 `~/.hermes/skills/` 目录，如果有超过 100 条 skill 且 50% 以上你自己看不出区别，那就是 "虚假产出和 patch 腐蚀"同时发作的状态。

&emsp;&emsp;`Curator` 不是 `Nudge` 的反向操作（删 skill），而是**质检环节**——它接受 `Nudge` 的产出，再做一次状态管理 + LLM 合并。可以把这套机制类比成"`Nudge` 是流水线工人不停往传送带上放零件，`Curator` 是巡检员每周来一次把次品挑出来归类"。下面我们打开它的源码看具体实现，第一站是触发机制——先理解 `Curator` 是**怎么被叫醒的**，才能理解后面 Phase 1/2 的全貌。

### 1.2 触发机制：idle-triggered，不是 cron daemon

&emsp;&emsp;打开 `agent/curator.py` 文件头注释（`curator.py:1-19`），我们会看到一段非常诚实的设计声明——**`The curator is an auxiliary-model task that periodically reviews agent-created skills and maintains the collection. It runs inactivity-triggered (no cron daemon)`**（中文："`Curator` 是一个辅助模型任务，周期性审查 agent 创建的 skill 库；它由 inactivity 触发，没有独立的 cron 守护进程"）。这句话点透了 `Curator` 的工作模式：**没有独立的后台守护进程**，没有 `crontab` 配置，没有 `systemd timer`。它是怎么定期跑的？答案是 **idle-triggered**（源码也写作 `inactivity-triggered`，含义相同）——agent 每次启动或处理 gateway tick 时，会调用一次 `should_run_now()` 检查，如果距离上次 `Curator` 运行已经超过 `interval_hours`（默认 7 天），且当前 agent 处于 idle 状态（最近 `min_idle_hours` 内没活动，默认 2 小时），就 fork 一个 background `Curator` agent 跑一次。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141441064.png" width=60%></div>

&emsp;&emsp;我们直接看源码。`curator.py:198-248` 是 `should_run_now()` 的完整实现，下面是教学节选——只保留闸门逻辑、省略 first-run seeding 的细节实现。

In [ ]:
# agent/curator.py:198-248 节选（伪代码 · 教学示意，完整版见源码）
# should_run_now: Curator 是否应该立即跑的"门禁"判断
def should_run_now(now: Optional[datetime] = None) -> bool:
    """返回 True 表示 curator 应当立即跑一次。

    Gates（闸门，全部通过才跑）:
      - curator.enabled == True            # 用户没禁用
      - not paused                         # 没被手动暂停
      - last_run_at present AND older than interval_hours  # 距上次跑超过间隔（默认 7 天）

    First-run 行为（首次见到 last_run_at 为空时）:
      - 不立即跑 → 把 now 写入 last_run_at 当 seed
      - 等满一个 interval 再触发第一次真实 run（防 fresh install 立刻乱动）
    """
    # 1. 静态 gate：用户配置闸
    if not is_enabled():
        return False
    if is_paused():       #用户显式执行 hermes curator pause，让 Curator 暂时不再自动触发
        return False

    # 2. 时间 gate：load 出上次运行时间
    state = load_state()
    last = _parse_iso(state.get("last_run_at"))
    if last is None:
        # 首次见到 → seed 一下，下次再判
        # （省略 seed 实现细节，见源码 L226-241）
        return False

    # 3. 间隔判断：现在距上次 ≥ interval_hours 才放行
    if now is None:
        now = datetime.now(timezone.utc)
    interval = timedelta(hours=get_interval_hours())  # 默认 7×24 = 168 小时
    return (now - last) >= interval

&emsp;&emsp;主代码块里 `L236-240` 被省略的 seed 实现单独拎出来看——这是 `should_run_now` 唯一一个"`return False` 但有副作用"的分支，值得放大镜单看：

In [ ]:
# agent/curator.py:226-241 节选——first-run seed 的真实实现
if last is None:
    if now is None:
        now = datetime.now(timezone.utc)
    try:
        state["last_run_at"] = now.isoformat()         # ← 关键这一行：把当前时间写进 state dict
        state["last_run_summary"] = (
            "deferred first run — curator seeded, "
            "will run after one interval"
        )
        save_state(state)                               # ← 写盘到 ~/.hermes/skills/.curator_state
    except Exception as e:
        logger.debug("Failed to seed curator last_run_at: %s", e)
    return False

&emsp;&emsp;这 5 行解释了"为什么 first-run 是个 `return False` 但下一次就能进入间隔判断"——`seed` 不是在内存里写，而是 `save_state(state)` 落盘到 `.curator_state` 文件。所以即使你 `Ctrl+C` 关掉 `hermes`，下次启动再 `load_state` 还能读到那个 seed 时间戳，不会陷入"每次都是 first run"的死循环。**注意：seed 动作发生在 `return False` 之前的同一次调用里，不是"等下次再写"——这是 `last_run_at` 被赋值的真实时刻。**另外 `try/except` 包住所有动作，seed 失败也只 `debug log`——`Curator` 这套"永不阻断主流程"的工程姿态在这里就开始体现了。

&emsp;&emsp;这段代码做了三件事。

- 第一，三道静态 gate（`enabled` / `not paused` / `last_run_at` 存在），任何一道不通过直接 `return False`。

- 第二，**首次运行行为**——`last_run_at` 为空时不立即跑，而是 seed 一个 `now` 进去，等满一个 `interval` 再触发第一次真实 review。这个设计很关键：避免 `hermes update` 后 background tick 立刻跑 `Curator` 把刚刚还没"跑热"的 skill 库乱动一通。

- 第三，**间隔判断**——`(now - last) >= interval` 才放行，间隔是 `interval_hours`（默认 168 小时即 7 天）。

&emsp;&emsp;另一个关键函数是 `maybe_run_curator()`（`curator.py:1656-1674`），这是真正被外部调用的入口——agent 启动时和 gateway tick 时都会调它一下：

In [ ]:
# agent/curator.py:1656-1674 节选（伪代码 · 教学示意，完整版见源码）
# maybe_run_curator: session-start hook + gateway tick 调用入口
def maybe_run_curator(
    *,
    idle_for_seconds: Optional[float] = None,
    on_summary: Optional[Callable[[str], None]] = None,
) -> Optional[Dict[str, Any]]:
    """Best-effort：所有 gate 都通过就跑一次 curator review。
    返回 result dict 表示真的跑了，None 表示被 gate 挡住了。永不抛异常。"""
    try:
        # 1. 静态闸先过：enabled / not paused / 时间间隔
        if not should_run_now():
            return None

        # 2. idle 闸：调用方提供 idle 时长才检查（默认要 idle ≥ 2 小时）
        if idle_for_seconds is not None:
            min_idle_s = get_min_idle_hours() * 3600.0
            if idle_for_seconds < min_idle_s:
                return None

        # 3. 全部通过 → 跑一次真实的 review
        return run_curator_review(on_summary=on_summary)
    except Exception as e:
        
        # 永不阻断主 agent 流程：失败只 debug log，不 raise
        logger.debug("maybe_run_curator failed: %s", e, exc_info=True)
        return None

&emsp;&emsp;这段代码体现了 `Curator` 的工程姿态——**永不阻断主 agent 流程**。`try/except` 包住一切，失败只写 debug log 不 raise；任何 gate 不通过就 silently `return None`。这意味着：跑 `hermes` 时不会因为 `Curator` 出问题而看到 error，最多看到 `hermes curator status` 显示"上次没跑"。

&emsp;&emsp;> **【常见误区】**：看到"`idle-triggered`"很容易理解成"我电脑空闲时就会跑"。实际上 `idle` 是指 **agent 进程的 idle**——不在用 `hermes` 跑 task 的时候。如果你 7 × 24 小时让 `hermes` 跑高负载任务，`Curator` 基本不会触发。判断自己是否被 gate 挡住的方法：跑 `hermes curator status`，看 `last_run_at` 字段——如果好几周前的时间还没刷新，那就是 `idle` 闸一直没过。

&emsp;&emsp;**Tier 1 验证：should_run_now 四道闸门测试**（仅 mock 不依赖 hermes 安装）

In [2]:
# 验证 should_run_now 的 4 道 gate 行为：enabled / paused / first-run-seed / interval
# 用 mock 替换 4 个外部依赖，避免依赖真实 hermes 安装
from datetime import datetime, timezone, timedelta
from unittest import mock

# Mock 的 5 个状态点：① 启用开关 ② 暂停标志 ③ 上次运行时间 ④ 配置间隔 ⑤ save_state 落盘
# 升级点：把 state 引用暴露出来 + 加 save_state mock，让测试 3 能验证 seed 真写入
def make_mocks(*, enabled=True, paused=False, last_run=None, interval_h=168):
    """构造 mock，模拟 4 道 gate 的不同状态组合 + 复刻 save_state 写盘行为。"""
    state = {"last_run_at": last_run.isoformat() if last_run else None}
    return {
        "is_enabled": lambda: enabled,
        "is_paused": lambda: paused,
        "load_state": lambda: state,
        "save_state": lambda s: state.update(s),  # ← 新增：复刻 save_state 把 dict update 回去
        "get_interval_hours": lambda: interval_h,
        "_state": state,                           # ← 新增：暴露 state 引用便于断言
    }

# 简化版 should_run_now（教学版，不引 hermes 实际依赖）
def should_run_now_test(mocks, now):
    """复刻 should_run_now 的 4 道 gate，全部通过才返回 True。"""
    if not mocks["is_enabled"]():
        return False, "gate1_disabled"
    if mocks["is_paused"]():
        return False, "gate2_paused"
    state = mocks["load_state"]()
    if state.get("last_run_at") is None:
        # ← 复刻源码 L226-241 的 seed 行为：写入 now 后再 return False
        state["last_run_at"] = now.isoformat()
        state["last_run_summary"] = "deferred first run — curator seeded"
        mocks["save_state"](state)
        return False, "gate3_first_run_seed"
    last = datetime.fromisoformat(state["last_run_at"])
    interval = timedelta(hours=mocks["get_interval_hours"]())
    if (now - last) < interval:
        return False, "gate4_interval_not_due"
    return True, "all_gates_passed"

# 4 个独立测试场景
now = datetime(2026, 5, 10, tzinfo=timezone.utc)

# 测试 1：用户禁用 → 不跑
ok, reason = should_run_now_test(make_mocks(enabled=False), now)
assert ok is False and reason == "gate1_disabled"
print(f"  [测试 1] 用户禁用 → 拒绝跑（reason={reason}）")

# 测试 2：暂停 → 不跑
ok, reason = should_run_now_test(make_mocks(paused=True), now)
assert ok is False and reason == "gate2_paused"
print(f"  [测试 2] 已暂停 → 拒绝跑（reason={reason}）")

# 测试 3：first-run（last_run_at 为空）→ 不跑 + seed 真写入（防 fresh install 立刻乱动）
mocks3 = make_mocks(last_run=None)
assert mocks3["_state"]["last_run_at"] is None, "调用前 seed 应当还没写"
ok, reason = should_run_now_test(mocks3, now)
assert ok is False and reason == "gate3_first_run_seed"
assert mocks3["_state"]["last_run_at"] == now.isoformat(), "调用后 seed 必须真写入"
print(f"  [测试 3] 首次运行 → 拒绝跑 + seed 真写入（last_run_at={mocks3['_state']['last_run_at'][:19]}）")

# 测试 4：last_run 是 3 天前，interval=7 天 → 间隔未到，不跑
ok, reason = should_run_now_test(make_mocks(last_run=now - timedelta(days=3)), now)
assert ok is False and reason == "gate4_interval_not_due"
print(f"  [测试 4] 3 天前跑过、间隔 7 天 → 拒绝跑（reason={reason}）")

# 测试 5：last_run 是 8 天前 → 间隔到 → 放行
ok, reason = should_run_now_test(make_mocks(last_run=now - timedelta(days=8)), now)
assert ok is True and reason == "all_gates_passed"
print(f"  [测试 5] 8 天前跑过、间隔 7 天 → 放行（reason={reason}）")

print("\n[OK] should_run_now 四道闸门 5 个分支全部覆盖")

  [测试 1] 用户禁用 → 拒绝跑（reason=gate1_disabled）
  [测试 2] 已暂停 → 拒绝跑（reason=gate2_paused）
  [测试 3] 首次运行 → 拒绝跑 + seed 真写入（last_run_at=2026-05-10T00:00:00）
  [测试 4] 3 天前跑过、间隔 7 天 → 拒绝跑（reason=gate4_interval_not_due）
  [测试 5] 8 天前跑过、间隔 7 天 → 放行（reason=all_gates_passed）

[OK] should_run_now 四道闸门 5 个分支全部覆盖


&emsp;&emsp;这 5 个测试覆盖了 `should_run_now` 的全部分支——三道静态 gate（enabled / paused / first-run-seed）+ 一道时间 gate 的两个方向（间隔未到 / 间隔已到）。**其中测试 3 不只断言 `return False`，还额外断言了 seed 后 `last_run_at` 真的被写入**——这就回答了"first-run 一句 `return False` 之后，`last_run_at` 什么时候才被赋值"的疑问：**就在这次调用本身里，`return` 之前的同一次执行内。** 任何一个 gate 不通过 `Curator` 就不会跑，这是 idle 触发机制能"温和不抢戏"的工程基础。

&emsp;&emsp;讲到这里我们已经在多处用到 `DEFAULT_INTERVAL_HOURS`、`DEFAULT_MIN_IDLE_HOURS` 这些常量，但还没正式介绍它们从哪来——本节末尾我们做一次小停顿，把"默认值机制"集中讲清楚，下面 1.3 节进入 Phase 1 状态机时还会用到 `DEFAULT_STALE_AFTER_DAYS / DEFAULT_ARCHIVE_AFTER_DAYS` 两个新常量，这套机制对它们同样适用。

&emsp;&emsp;**学员在自己机器验证的第一道坑**——打开 `~/.hermes/config.yaml` 很可能找不到 `curator:` 块。这并不代表 `hermes` 版本旧没装 `Curator`，而是源码用了"配置文件不写就走 default 常量"的兜底机制。先看四个默认值常量本体（`curator.py:56-59`）：

In [ ]:
# agent/curator.py:56-59 节选——四个默认值常量
DEFAULT_INTERVAL_HOURS     = 24 * 7   # 168 小时 = 7 天，should_run_now 的时间闸
DEFAULT_MIN_IDLE_HOURS     = 2        # 2 小时，maybe_run_curator 的 idle 闸
DEFAULT_STALE_AFTER_DAYS   = 30       # 30 天，Phase 1 active → stale 阈值（1.3 节用到）
DEFAULT_ARCHIVE_AFTER_DAYS = 90       # 90 天，Phase 1 stale → archived 阈值（1.3 节用到）

&emsp;&emsp;这四行是整个 `Curator` 时间逻辑的"原点"——所有"7 天、2 小时、30 天、90 天"的数字都从这里出。下面看怎么读取它们（`curator.py:148-167`，4 个 getter 用同一套"配置缺省回退"模式）：

In [ ]:
# agent/curator.py:148-167 节选——"配置缺省回退"统一模式
def _load_config() -> Dict[str, Any]:
    """读 ~/.hermes/config.yaml 的 curator: 块；文件不存在 / 块不存在均返回 {}"""
    try:
        from hermes_cli.config import load_config
        cfg = load_config()
    except Exception:
        return {}
    cur = cfg.get("curator") or {}           # ← 缺 curator: 块 → 空 dict 兜底
    return cur if isinstance(cur, dict) else {}

def is_enabled() -> bool:
    return bool(_load_config().get("enabled", True))           # ← 缺 key → 默认 True

def get_interval_hours() -> int:
    try:
        return int(_load_config().get("interval_hours", DEFAULT_INTERVAL_HOURS))
    except (TypeError, ValueError):
        return DEFAULT_INTERVAL_HOURS                          # ← 类型错也兜底

def get_min_idle_hours() -> float:
    try:
        return float(_load_config().get("min_idle_hours", DEFAULT_MIN_IDLE_HOURS))
    except (TypeError, ValueError):
        return DEFAULT_MIN_IDLE_HOURS

&emsp;&emsp;关键在 `cfg.get("xxx", DEFAULT_XXX)` 这种 dict 默认值语法——Python 的 `dict.get(key, default)` 找不到 key 时返回 `default`，从不抛异常。配合外层 `try/except` 兜住"类型错误的配置"（譬如有人写了 `interval_hours: "seven"`），形成三层兜底：

① 整个 yaml 文件不存在 
② `curator:` 块不存在 
③ 单个 key 不存在或类型错。任何一层失败都回退到 `DEFAULT_*` 常量。**所以"`config.yaml` 没有 `curator:` 块"≡ "全套默认值生效"，不是"功能没启用"。**

&emsp;&emsp;四个默认值与覆盖位置一张表打尽：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 1-3 Curator 四个默认值常量速查（无配置即生效）</font></p>
<div class="center">

| 默认值常量 | 源码位置 | 默认值 | 控制的闸门 | 覆盖路径（config.yaml） |
|---|---|---|---|---|
| `DEFAULT_INTERVAL_HOURS` | `curator.py:56` | `168`（7 天）| Gate 4 时间闸 | `curator.interval_hours` |
| `DEFAULT_MIN_IDLE_HOURS` | `curator.py:57` | `2`（小时）| idle 闸（生产路径传 `inf` 已旁路） | `curator.min_idle_hours` |
| `DEFAULT_STALE_AFTER_DAYS` | `curator.py:58` | `30`（天）| Phase 1 `active→stale` 阈值 | `curator.stale_after_days` |
| `DEFAULT_ARCHIVE_AFTER_DAYS` | `curator.py:59` | `90`（天）| Phase 1 `stale→archived` 阈值 | `curator.archive_after_days` |

</div>


&emsp;&emsp;想覆盖默认值时，在 `~/.hermes/config.yaml` 加这样一段（任意子集，未写的字段自动回退到默认）：

```yaml
curator:
  enabled: true              # 默认 True；要完全关掉 Curator 就改 false
  interval_hours: 24         # 改成每天跑一次（默认 168 = 一周一次）
  min_idle_hours: 0.5        # idle 闸放宽到 30 分钟（默认 2 小时）
  stale_after_days: 30       # 显式声明，跟默认值一致
  archive_after_days: 90     # 显式声明，跟默认值一致
```

&emsp;&emsp;> **【常见误区】**：很多学员第一次看 `Curator` 课件、回到自己机器打开 `~/.hermes/config.yaml` 发现没有 `curator:` 块，第一反应是"`hermes` 版本旧没装这个功能"。**实际上这是默认值机制——`hermes` 团队故意让 `Curator` 所有配置都"零配置直接生效"**，让大多数用户不需要任何手动配置就能享受周自动整理。判断自己机器上 `Curator` 是否真在跑的正确方法是：① `cat ~/.hermes/skills/.curator_state` 看 `last_run_at` 和 `run_count` 字段；② `hermes curator status` 命令直接读这个文件并人类可读地展示。如果文件存在且 `run_count > 0`，说明 `Curator` 已经在你机器上至少跑过一次真实 review；如果 `last_run_at` 有值但 `run_count == 0`，说明刚 seed 完、还在等首个 interval 走完。

### 1.3 Phase 1：自动 Transition——纯函数无 LLM 的状态机

&emsp;&emsp;`Curator` 跑起来后做的第一件事是 **Phase 1 自动 transition**。这是个**纯函数**——完全不调用 LLM，只看时间戳，按 30 天 / 90 天阈值把 skill 状态在 `active / stale / archived` 之间迁移。源码在 `curator.py:255-295`，下面是完整教学节选：

In [ ]:
# agent/curator.py:255-295 节选（伪代码 · 教学示意，完整版见源码）
# apply_automatic_transitions: active → stale → archived 三态自动迁移
def apply_automatic_transitions(now: Optional[datetime] = None) -> Dict[str, int]:
    """遍历所有 agent-created skill，按最近活动时间戳迁移状态。
    Pinned skill 在 Curator 中跳过（不参与状态迁移）。返回各类计数 dict。"""
    from tools import skill_usage as _u

    if now is None:
        now = datetime.now(timezone.utc)
    # 两个时间阈值：超过 30 天没用 → stale；超过 90 天没用 → archived
    stale_cutoff = now - timedelta(days=get_stale_after_days())     # 默认 30 天
    archive_cutoff = now - timedelta(days=get_archive_after_days()) # 默认 90 天

    counts = {"marked_stale": 0, "archived": 0, "reactivated": 0, "checked": 0}

    # 关键：只扫 agent-created 的 skill（agent_created_report 已经按 created_by 过滤）
    for row in _u.agent_created_report():
        counts["checked"] += 1
        name = row["name"]
        # 五条护栏之一：pinned skill 在 Curator 中跳过，不参与 transition
        if row.get("pinned"):
            continue

        # 取"最近一次活动时间戳"（从未 active 过则用 created_at 兜底）
        last_activity = _parse_iso(row.get("last_activity_at"))
        anchor = last_activity or _parse_iso(row.get("created_at")) or now
        if anchor.tzinfo is None:
            anchor = anchor.replace(tzinfo=timezone.utc)

        current = row.get("state", _u.STATE_ACTIVE)

        # 状态机三档判断：archive 优先于 stale，stale 优先于 reactivate
        if anchor <= archive_cutoff and current != _u.STATE_ARCHIVED:
            # 超过 90 天 → archive（移到 ~/.hermes/skills/.archive/，不删！）
            ok, _msg = _u.archive_skill(name)
            if ok:
                counts["archived"] += 1
        elif anchor <= stale_cutoff and current == _u.STATE_ACTIVE:
            # 超过 30 天 + 当前还是 active → 标 stale
            _u.set_state(name, _u.STATE_STALE)
            counts["marked_stale"] += 1
        elif anchor > stale_cutoff and current == _u.STATE_STALE:
            # 30 天内被重新使用 + 当前是 stale → reactivate 回 active
            _u.set_state(name, _u.STATE_ACTIVE)
            counts["reactivated"] += 1

    return counts

&emsp;&emsp;这段代码的核心逻辑可以拆成四件事：

&emsp;&emsp;**第一件：只扫 agent-created**。`for row in _u.agent_created_report()` 这一行已经把"用户手写的 skill"全部过滤掉了，根本不进入循环——这正是第 2 章要讲的 `ContextVar` 血统隔离的源头。

&emsp;&emsp;**第二件：pinned 跳过**（`curator.py:271`）。任何被 pin 的 skill，无论多久没用都跳过不动（不进状态机，Phase 2 prompt 也明确排除）。注意：pin 只影响 Curator 的整理行为，`skill_manage` 侧的 patch/edit 等操作不受此限制。

&emsp;&emsp;**第三件：状态机三档**。`active → stale → archived` 单向迁移（30 天没用标 stale，90 天没用 archive），但 `stale → active` 可以反向——30 天内重新使用就 reactivate，避免错杀有价值的旧 skill。

&emsp;&emsp;**第四件：archive 不是 delete**。`_u.archive_skill(name)` 只是把目录移到 `~/.hermes/skills/.archive/`，文件还在，可以恢复。这是整套护栏的核心铁律：`Never auto-deletes — only archives. Archive is recoverable`（文件头注释原文）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141442985.png" width=50%></div>

&emsp;&emsp;这套状态机的设计意图非常明确：**用时间窗口做粗筛**——长时间没用的 skill 优先级低，可以先标 stale 让 Phase 2 LLM Review 重点处理；再长时间没用就 archive 出主目录腾位置，但保留可恢复路径。整个 Phase 1 不调用任何 LLM，是纯函数 + 数据库读写——这意味着即使 `Curator` review 阶段（Phase 2）的 LLM 调用失败，Phase 1 已经做完了基础整理。

&emsp;&emsp;为了让这套机制可验证，下面用一段教学示意代码确认 `curator.py` 可以 import 且核心阈值配置可读（**Tier 1 验证 cell**，在 Demo 1 之前先确认环境就绪）。

In [3]:
# Tier 1 验证：确认 curator.py 可读，核心阈值配置常量可见
# 注意：这段代码需要本地有 hermes-agent 仓库，或把仓库路径加入 sys.path
import sys
import os

# 把本地 HermesAgent 仓库加入 import 路径（HERMES_REPO_PATH 来自 第 0.4 节 .env）
repo = os.getenv("HERMES_REPO_PATH", "/Users/your_name/Git/HermesAgent")
if os.path.isdir(repo) and repo not in sys.path:
    sys.path.insert(0, repo)

try:
    # 直接 import curator 模块（不会触发任何实际 review）
    from agent import curator
    # 打印三个核心默认阈值，让你看到"7 天 / 30 天 / 90 天"是从哪儿来的
    print(f"[OK] curator 模块加载成功")
    print(f"  DEFAULT_INTERVAL_HOURS    = {curator.DEFAULT_INTERVAL_HOURS} 小时（{curator.DEFAULT_INTERVAL_HOURS // 24} 天）")
    print(f"  DEFAULT_MIN_IDLE_HOURS    = {curator.DEFAULT_MIN_IDLE_HOURS} 小时")
    print(f"  DEFAULT_STALE_AFTER_DAYS  = {curator.DEFAULT_STALE_AFTER_DAYS} 天")
    print(f"  DEFAULT_ARCHIVE_AFTER_DAYS= {curator.DEFAULT_ARCHIVE_AFTER_DAYS} 天")
except ImportError as e:
    # 本地没装 hermes 也不阻断 Demo——后面会给 LangChain 模拟版
    print(f"[INFO] curator 模块未加载：{e}")
    print("  （这正常——Demo 1 末尾会给你一个 LangChain 模拟版，不依赖 hermes）")

[INFO] curator 模块未加载：No module named 'agent'
  （这正常——Demo 1 末尾会给你一个 LangChain 模拟版，不依赖 hermes）


&emsp;&emsp;运行这段代码若看到 4 行 `DEFAULT_*` 输出，说明本地源码可读。这 4 个默认值（`7 days / 2 hours / 30 days / 90 days`）就是 `Curator` 整套时间窗口的来源——可以在 `~/.hermes/config.yaml` 的 `curator:` 块里覆盖它们（参见 1.2 节末"默认值机制"小节给的覆盖示例），但默认值就足够生产用。如果想"提前试跑一次 `Curator`"，调小 `interval_hours`（譬如改成 `1`）就能不用等满 7 天就触发。

&emsp;&emsp;Phase 1 讲完，下面进入 Phase 2 LLM Review。

### 1.4 Phase 2：LLM Review——Umbrella 合并的 fork agent

&emsp;&emsp;Phase 1 做完后，`Curator` 进入 Phase 2。这一阶段 fork 一个 review agent（用 auxiliary 辅助模型，不占主 session 的 prompt cache），让它读 Phase 1 整理后的 skill 列表，做 LLM 层的合并 / 降级 / archive 决策。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141443006.png" width=70%></div>

&emsp;&emsp;Phase 2 的核心是一段很长的 prompt（源码在 `curator.py:329-440`），关键部分是 **Umbrella 概念** 和 **5 条 hard rules**：

In [ ]:
# agent/curator.py:329-358 节选（伪代码 · 教学示意，完整版见源码）
# Phase 2 prompt：告诉 fork 出来的 review agent 它的工作目标和硬约束
CURATOR_REVIEW_PROMPT = (
    "You are running as Hermes' background skill CURATOR. This is an "
    "UMBRELLA-BUILDING consolidation pass, not a passive audit and not a "
    "duplicate-finder.\n\n"

    # 核心理念：skill 库的目标是"class-level 大伞 skill"，不是"一会话一窄 skill"
    "The goal of the skill collection is a LIBRARY OF CLASS-LEVEL "
    "INSTRUCTIONS AND EXPERIENTIAL KNOWLEDGE. A collection of hundreds of "
    "narrow skills where each one captures one session's specific bug is "
    "a FAILURE of the library — not a feature.\n\n"

    # 4 条硬约束 hard rules
    "Hard rules — do not violate:\n"
    "1. DO NOT touch bundled or hub-installed skills. The candidate list "
    "below is already filtered to agent-created skills only.\n"
    # ↑ 护栏 1：bundled / hub-installed skill 完全不动
    "2. DO NOT delete any skill. Archiving (moving the skill's directory "
    "into ~/.hermes/skills/.archive/) is the maximum destructive action. "
    "Archives are recoverable; deletion is not.\n"
    # ↑ 护栏 2：永远不删，archive 是最大破坏性动作
    "3. DO NOT touch skills shown as pinned=yes. Skip them entirely.\n"
    # ↑ 护栏 3：pinned，Curator 跳过
    "4. DO NOT use usage counters as a reason to skip consolidation. The "
    "counters are new and often mostly zero. Judge overlap on CONTENT, "
    "not on use_count. 'use=0' is not evidence a skill is valuable; it's "
    "absence of evidence either way.\n"
    # ↑ 护栏 4：判断 overlap 看内容不看 use_count（防"我没用过所以不能动"的错判）
    "5. DO NOT reject consolidation on the grounds that 'each skill has "
    "a distinct trigger'. Pairwise distinctness is the wrong bar.\n"
    # ↑ 护栏 5：不靠 'distinct trigger' 拒绝合并——多窄 skill 用一个带子分节的 umbrella 更好
    # （省略后续 How to work、3 种合并方式、toolset 说明）
)

&emsp;&emsp;这段 prompt 有两个关键设计点。

&emsp;&emsp;**设计点一：`UMBRELLA-BUILDING` 核心概念**。`Curator` 不是去重器，不是被动审计器，而是"聚合器"——按 "cluster + merge" 指令把一堆窄 skill（如 `pr-triage-feature-A/B/C` 三条）合并成一个 `pr-triage` 大伞 skill，原来的 sibling 内容降级为 `pr-triage/references/` 下的支持文件。目的是让 skill 库保持"少而广"，而不是"多而窄"。

&emsp;&emsp;**设计点二：5 条 hard rules 全部围绕"少做、不做"**。① 不动 bundled；② 不删任何东西；③ 不动 pinned；④ 不靠 use_count 做判断；⑤ 不靠 distinct trigger 拒绝合并。每多一条 hard rule，就少一类做错可能。

&emsp;&emsp;Phase 2 完整的工作链路是这样的（见 `curator.py:1322-1332` 区域）：先做 `snapshot_skills()` 备份当前 skill 库整体快照（`pre-curator-run`），然后调 `apply_automatic_transitions` 跑 Phase 1，再 fork 一个用 `auxiliary` 模型的 review agent 跑 Phase 2 prompt，让它通过 `skill_manage` 工具做实际的 patch / create / archive。整个过程只动 `agent-created` skill，不污染主 session 的 prompt cache（这是 `curator.py` 文件头注释里的第四条 strict invariant）。

&emsp;&emsp;**Phase 2 运行时 contract**（fork review agent 的关键参数和约束，源码 `curator.py:1380-1420` 区间）：

<p align="center">表 1-2 Phase 2 LLM Review 运行时关键参数</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 参数 / 约束 | 默认值 / 行为 | 工程含义 |
| :---: | :---: | :---: |
| `max_iterations` | `9999`（当前用户文档写 8，源码实际为 9999） | 给 review agent 充分迭代空间，但有 token 预算兜底 |
| `quiet_mode` | `True` | review agent 内部不打印日志，避免污染主 session 输出 |
| `skip_context_files` | `True` | 不加载主 session 的 context files，避免话题污染 |
| `skip_memory` | `True` | 不读 `MEMORY.md / USER.md`，避免被主用户 context 影响 review 决策 |
| **防递归** | review agent 内部不再 fork review agent | `Curator` 不会嵌套触发 `Curator` |
| structured summary | `consolidated` / `pruned` 字段 | 真实改动 vs 跳过的 skill 在结构化输出中分开记账，便于复盘 |

&emsp;&emsp;这套 contract 是 Phase 2 能"安全 fork"的工程基础——`skip_context_files` + `skip_memory` 让 review agent 在干净的上下文里做合并决策；防递归保证 `Curator` 不会嵌套触发自己；structured summary 让用户能在事后通过 `last_report_path` 复盘"这次合并了哪些 / 跳过了哪些"。

&emsp;&emsp;**Phase 2 模型路由：三层优先级**（源码 `curator.py:1450-1577`）：

In [ ]:
# 伪代码 · 教学示意：_resolve_review_runtime 的三层路由优先级
# 1. 优先：auxiliary.curator.{provider, model, api_key, base_url}（专门为 curator 配置的辅助模型）
# 2. fallback：legacy curator.auxiliary（旧版兼容路径）
# 3. 兜底：main agent model（主模型，最贵但保证可跑）

&emsp;&emsp;这是 hermes "辅助模型"的标准路由模式——`Curator` 只是这套通用路由的一个使用者。配置时优先指定 `auxiliary.curator` 让 review agent 用便宜的小模型（譬如 `gpt-4o-mini`）跑 prompt 优化，省 token；不配的话自动 fallback 到主模型。

&emsp;&emsp;> **【常见误区】**：你看到 "umbrella" 这个词容易理解成"一个大目录"。其实 `umbrella` 在 `Hermes` 语境里是 **一个 SKILL.md**——一个写得足够泛、足够 class-level 的 SKILL.md，下面挂 `references/` `templates/` `scripts/` 三个支持目录。区别在于：`umbrella` skill 本身仍然是一条可以被 agent 通过描述召回的 skill，只是它的 SKILL.md body 涵盖一类工作流而不是某个具体 case。

&emsp;&emsp;到这里我们已经看完了 `Curator` 双 Phase 的完整工作流——Phase 1 纯函数兜底、Phase 2 LLM 做 umbrella 合并，配合 contract（quiet / skip_memory / 防递归）和模型路由（三层 fallback）让 fork 安全可控。下一节汇总五条护栏，看清保护 skill 库安全的完整边界。

### 1.5 五条护栏：永远不删 / 不动 bundled / pinned 跳过 / snapshot / 血统

&emsp;&emsp;前面三节把 `Curator` 的完整工作流走了一遍。本节把散落在源码各处的护栏机制收束成一张表，看清整套安全设计的全貌。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513172247519.png" width=60%></div>

&emsp;&emsp;展开顺序：先用一段代码证明 `archive ≠ delete`（Tier 1 验证），再用表 1-1 汇总五条护栏，最后说明第 5 条"血统隔离"为什么要单独放到第 2 章讲。

&emsp;&emsp;**Tier 1 验证：archive 不是 delete——护栏 1 的代码证明**（用 tempfile 真实文件系统）

In [30]:
# 验证 archive 真把 skill 目录移到 .archive/，原内容完整保留——这是"永远不删"承诺的代码证明
import tempfile
import shutil
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    skills_dir = tmp / "skills"
    archive_dir = skills_dir / ".archive"
    skills_dir.mkdir()
    archive_dir.mkdir()

    # 模拟一条 agent-created skill（含 SKILL.md 主体 + references/ 支持目录）
    skill_name = "old-jira-sync"
    skill_path = skills_dir / skill_name
    skill_path.mkdir()
    (skill_path / "SKILL.md").write_text("# Old Jira Sync\n业务说明 ...\n", encoding="utf-8")
    refs_dir = skill_path / "references"
    refs_dir.mkdir()
    (refs_dir / "schema.md").write_text("# Schema 字段说明\n", encoding="utf-8")

    # 验证前置条件：skill 目录存在 + archive 目录为空
    assert skill_path.exists() and (skill_path / "SKILL.md").exists()
    assert not (archive_dir / skill_name).exists()
    print(f"  [前置] skill 目录存在: {skill_path.exists()}")
    print(f"  [前置] archive 目录为空: {len(list(archive_dir.iterdir())) == 0}")

    # 模拟 archive_skill：把目录从 skills/<name> 移到 skills/.archive/<name>
    # 真 hermes 用 shutil.move（agent/skill_usage.py），这里复刻关键动作
    archived_path = archive_dir / skill_name
    shutil.move(str(skill_path), str(archived_path))

    # 验证 archive 后置条件
    # 关键断言 1：原位置消失
    assert not skill_path.exists(), "原 skill 目录应该消失"
    # 关键断言 2：archive 位置存在 + 文件完整
    assert archived_path.exists()
    assert (archived_path / "SKILL.md").exists()
    assert (archived_path / "SKILL.md").read_text(encoding="utf-8").startswith("# Old Jira Sync")
    # 关键断言 3：references 子目录也完整跟过去
    assert (archived_path / "references" / "schema.md").exists()
    print(f"  [后置] 原位置消失: {not skill_path.exists()}")
    print(f"  [后置] archive 位置内容完整: SKILL.md + references/schema.md 都在")

print("\n[OK] archive 不是 delete——目录真的移到 .archive/，所有内容完整保留")
print("[OK] 验证了护栏 1：archive 可恢复（用户手动 unarchive 或 hermes 命令都行）")

  [前置] skill 目录存在: True
  [前置] archive 目录为空: True
  [后置] 原位置消失: True
  [后置] archive 位置内容完整: SKILL.md + references/schema.md 都在

[OK] archive 不是 delete——目录真的移到 .archive/，所有内容完整保留
[OK] 验证了护栏 1：archive 可恢复（用户手动 unarchive 或 hermes 命令都行）


&emsp;&emsp;这个测试用 `tempfile.TemporaryDirectory` 创建真实磁盘目录、模拟一条带 `references/` 子目录的 skill，然后用 `shutil.move`（hermes 内部 `archive_skill()` 的核心动作）把它移到 `.archive/`。三条断言验证：① 原位置真的消失；② `.archive/<name>/SKILL.md` 存在且内容完整；③ `references/` 子目录也完整跟过去。这是"永远不删 + archive 可恢复"承诺的**代码层证明**——不是计数器层面的 `counts["archived"] == 1`，而是磁盘事实。

&emsp;&emsp;下面汇总 `Curator` 整套设计的五条核心护栏（保护 skill 库安全）。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 1-1 Curator 五条护栏汇总（按"防错严重程度"排序）</font></p>
<div class="center">

| # | 护栏 | 实现位置 | 防错效果 |
|---|---|---|---|
| 1 | 永远不删（archive 是最大破坏性动作）| `curator.py:17` 文件头 strict invariant + `CURATOR_REVIEW_PROMPT` rule 2 | archive 可恢复，删除不可恢复——给"误判"留一条退路 |
| 2 | 不动 `bundled` / `hub-installed` skill | `agent_created_report()` 已过滤 + `CURATOR_REVIEW_PROMPT` rule 1 | 系统级预装 skill 完全不进 candidate list |
| 3 | `pinned` skill Curator 跳过 | `curator.py:271` `if row.get("pinned"): continue` + prompt rule 3 | 用户标记"必须保留"的 skill 不进 Curator 状态机 |
| 4 | 启动前 `snapshot_skills`（best-effort）| `curator.py:1322` `snapshot = curator_backup.snapshot_skills(reason="pre-curator-run")` | 非 dry-run 才执行；快照失败不阻断主流程；不等于事务级回滚——是"尽力而为的备份"，不是硬保证 |
| 5 | 血统隔离：只动 `created_by == "agent"` | `agent_created_report()` 内部按 `created_by` 过滤 + 第 2 章 `ContextVar` 机制保证标记准确 | 按当前源码/tests，foreground 用户 skill 不进入 `Curator` 候选 |

</div>


&emsp;&emsp;读这张表的姿势：五条护栏里四条是"做减法"（不删 / 不动 bundled / 不动 pinned / 只动 agent-created），一条是"做加法"（启动前 snapshot）。**这些护栏降低了后台任务误伤用户资产的风险**——正因为 `Curator` 在删除动作上极度保守，`hermes` 才能把它默认开启，用户才敢让它一直跑着。

&emsp;&emsp;另一个值得注意的细节是 **第 5 条护栏**——它依赖第 2 章要讲的 `ContextVar` 血统隔离机制。在 5 月之前的 `Hermes`，所有通过 `skill_manage(create)` 创建的 skill 都标记为 agent-created，这导致用户用 `agent` 工作流创建的 skill 也被 `Curator` 整理（`Reddit` 上"`It will overwrite your edits`"（"它会覆盖你的修改"）那条吐槽就是这么来的）。5 月新增的 `ContextVar` 隔离精确区分了"前台用户驱动调用"和"后台 review agent 调用"，让 `Curator` 真正只动 agent 自己写的 skill。这是第 2 章要展开的内容。

#### 1.5.1 Demo 1：Curator Dry-Run（10 min）

&emsp;&emsp;`Demo 1` 的目标是让我们**亲眼看到 `Curator` 识别哪些 skill 需要整理、提出了哪些操作建议**——但**不真的执行**这些操作（dry-run 模式）。这个 demo 提供两条路径：路径 A 适合本地真有 `hermes` 安装的同学，调用 CLI 跑（3-5 min，无 API 费用）；路径 B 适合没安装 `hermes` 的同学，用 `LangChain` + `DeepSeek` 完整模拟 `Curator` 双 Phase 闭环（8-12 min，预计 ¥0.1-0.2 token 费）。两条路径都能让你理解 `Curator` 的 Phase 1 状态过滤 + Phase 2 LLM umbrella 合并是怎么生效的。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141442491.png" width=50%></div>

&emsp;&emsp;**步骤一：路径 A（本地有 hermes 安装）—— `hermes curator run --dry-run`**

&emsp;&emsp;如果本地装了 `hermes` 并跑了一段时间（`~/.hermes/skills/` 下有 agent-created 的 skill），可以直接调 CLI 看 `Curator` 的判断：

In [4]:
# 路径 A：调 hermes CLI 跑 dry-run（不会真改任何 skill）
# dry-run 模式下：Phase 1 会 print 出"会做什么"但不真的 set_state；Phase 2 会 fork agent 但不真的 patch
import subprocess

# 步骤 1：先看当前 curator 状态——上次跑的时间、是否被禁用、skill 总数等
# subprocess.run 用 capture_output 捕获 stdout/stderr，timeout 防卡死
result = subprocess.run(
    ["hermes", "curator", "status"],   # CLI 命令：查询 curator 当前状态
    capture_output=True, text=True, timeout=30
)
# 打印完整 stdout 让你看到 last_run_at / skill 计数 / paused 标记
print("=== hermes curator status ===")
print(result.stdout)
# 步骤 2：如果命令失败（hermes 未安装 or PATH 没配），fallback 到路径 B
if result.returncode != 0:
    print(f"[FAIL] exit={result.returncode}")
    print(f"  stderr: {result.stderr}")
    print('  → 你可能还没装 hermes，跳到下面"路径 B"用 LangChain + DeepSeek 模拟版')

=== hermes curator status ===
curator: ENABLED
  runs:           2
  last run:       4d ago
  last summary:   dry-run auto: no changes; llm: The data is clear. There is exactly **1 agent-created skill**: `web-content-curation`. It is:
                  - **Not pinned** (`pinned=no`)
                  - **Active** (`state=active`)
                  - Already a **well-structured class-level umbrella skill** with labeled subsections a…
  last report:    /Users/mac/.hermes/logs/curator/20260513-083916
  interval:       every 7d
  stale after:    30d unused
  archive after:  90d unused

agent-created skills: 1 total
  active     1
  stale      0
  archived   0

least recently active (top 5):
  web-content-curation                      activity= 14  use=  6  view=  6  patches=  2  last_activity=3h ago

most active (top 5):
  web-content-curation                      activity= 14  use=  6  view=  6  patches=  2  last_activity=3h ago

least active (top 5):
  web-content-curation               

&emsp;&emsp;`hermes curator status` 输出会告诉你：上次 `Curator` 运行时间、当前 skill 总数、有几条 stale、有几条 archived、是否被 paused。这是你了解"我的 skill 库现在长什么样"的第一手数据——如果你跑了一段时间 `hermes` 但 `last_run_at` 显示从未运行过，那很可能你的使用模式没满足 `idle-triggered` 闸门（譬如你的 idle 时长一直不到 2 小时）。

&emsp;&emsp;接下来跑 `dry-run`。运行后会看到三类输出：① `Phase 1` 触发情况和扫描的 skill 总数；② `Phase 2` 是否真的 fork 了 review agent；③ 各 skill 的状态机迁移建议（哪些会被 archive、哪些 stale 转 active）。如果输出里看到 `Curator already ran within X hours` 而提前退出，说明 idle 闸门没满足，等几小时再跑或调小 `min_idle_hours`。

In [5]:
# 跑 dry-run：让 Curator 实际扫一遍并打印建议，但不写状态
result = subprocess.run(
    ["hermes", "curator", "run", "--dry-run"],
    capture_output=True, text=True, timeout=300  # 给 5 分钟，因为 Phase 2 LLM 可能慢
)
print("=== hermes curator run --dry-run ===")
print(result.stdout[:])  # 只打印最后 2000 字符，避免输出过长
print(f"\n[exit code] {result.returncode}")

=== hermes curator run --dry-run ===
curator: running DRY-RUN (report only, no mutations)...
curator: dry-run auto: no changes; llm: ## Curator Dry-Run Report: Umbrella-Building Consolidation

### Candidate Pool Analysis

Only **one** agent-created skill is present in the library:

| Skill | State | Pinned | Activity | Use | View | Patches |
|-------|-------|--------|---…
auto (preview): 1 candidate skill(s) — no transitions applied in dry-run
dry-run: no changes applied. Read the report with `hermes curator status` and run `hermes curator run` (no flag) to apply.


[exit code] 0


&emsp;&emsp;`dry-run` 输出有几个关键信号要观察：① `Phase 1 summary`——skill 库里多少条被检查、多少条被标 stale、多少条被 archive；② `Phase 2 LLM Review`——fork 出来的 review agent 用了多少 token、给你提了几个 umbrella 合并建议；③ `report file location`——生成的报告目录（默认 `~/.hermes/logs/curator/{YYYYMMDD-HHMMSS}/`），里面有 `REPORT.md`（人类可读的逐 skill 决策记录）+ `run.json`（结构化运行数据）+ 可能的 `cron_rewrites.json`，建议打开 `REPORT.md` 逐条读一遍。`hermes curator status` 的 `last report:` 字段会显示最新一次报告的目录。**观察点**：哪些 skill 被标 stale？LLM 提了哪些 umbrella 合并？有没有出现"误标"（譬如你昨天还在用的 skill 被标 stale）？如果出现误标，记录下来——这是给社区反馈或自己调阈值的依据。

&emsp;&emsp;**步骤二：路径 B（没装 hermes）—— LangChain 模拟完整 Curator 双 Phase 闭环**

&emsp;&emsp;如果本地没装 `hermes`，下面这段 LangChain 模拟版会让我们看到 `Curator` **完整工作流的两个 Phase**——不是只跑 Phase 1 纯函数状态机，而是把 Phase 1 输出作为 Phase 2 LLM Review 的输入，**亲手让 LLM 给出 umbrella 合并决策**。这跟真 `Curator` 的设计是 1:1 同构的：Phase 1 按时间窗口过滤候选 → Phase 2 fork 一个受限 `review_agent` 做 LLM 决策。我们用 LangChain 的 `create_agent` + `@tool` 把这套机制工业化复刻一遍，每行代码都能对应 `curator.py` 的源码段。

&emsp;&emsp;**B.1 准备工作**——`.env` 文件里要有 `DEEPSEEK_API_KEY` 和 `DEEPSEEK_MODEL`（参考第 0.4 节），并装好 LangChain 三件套：

In [6]:
# ── B.1 环境准备 ──────────────────────────────────────────────────
# 依赖已在 §0.4 requirements.txt 安装，这里无需重新 pip install
# .env 也已在 §0.4 加载，这里直接读取环境变量即可

import os

# 读取模型名称（默认 deepseek-chat，可在 .env 中的 DEEPSEEK_MODEL 覆盖）
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")

# 确认 API Key 已配置（路径 B 需要真实调用 DeepSeek API）
assert os.getenv("DEEPSEEK_API_KEY"), "请先在 .env 配置 DEEPSEEK_API_KEY，参考 §0.4"

print(f"[OK] DEEPSEEK_MODEL = {DEEPSEEK_MODEL}")

[OK] DEEPSEEK_MODEL = deepseek-chat


&emsp;&emsp;**B.2 Mock skill 库 + Phase 1 预过滤**——这次我们的 5 条 mock skill 不是随意挑的，而是**刻意设计成"3 条同主题碎片化窄 skill + 1 条独立 active + 1 条 pinned"**——让 Phase 2 LLM 真的有 umbrella 合并的机会可演：

In [7]:
# B.2 Mock skill 库 + Phase 1 纯函数预过滤
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
mock_skills = [
    # ── 一组 umbrella 候选：pr-feedback-* 系列三条窄 skill（同主题碎片化）──
    {"name": "pr-feedback-rebase",            "days_idle": 5,  "state": "active",
     "summary": "处理 reviewer 提的 rebase 冲突；逐 commit cherry-pick 路径"},
    {"name": "pr-feedback-conflict-resolve",  "days_idle": 8,  "state": "active",
     "summary": "PR 合并冲突的逐文件处理；优先 incoming / current 选择策略"},
    {"name": "pr-feedback-reviewer-comments", "days_idle": 12, "state": "active",
     "summary": "回复 reviewer 评论的标准格式；附 diff 片段 + 解释"},
    # ── 一条独立 active skill（无合并对象，应被 skip）──
    {"name": "kafka-consumer-group-debug",    "days_idle": 6,  "state": "active",
     "summary": "kafka rebalance 频繁触发的根因排查；session.timeout 调优"},
    # ── 一条 pinned 关键 skill（护栏 3 保护，Phase 1 直接过滤掉）──
    {"name": "critical-prod-runbook",         "days_idle": 180, "state": "active", "pinned": True,
     "summary": "生产事故 P0 响应手册（pinned）"},
]
# 默认 pinned=False
for s in mock_skills:
    s.setdefault("pinned", False)

# Phase 1 纯函数过滤：跳过 pinned（对应 curator.py:271 if row.get("pinned"): continue）
# 真 Curator 这里还会做 30/90 天状态机迁移，本 demo 简化只保留 pinned 过滤
# —— 因为我们的所有 mock skill 都在 30 天活跃窗口内，本身不触发 stale/archive
candidates = [s for s in mock_skills if not s["pinned"]]

print(f"=== Phase 1 预过滤 ===")
print(f"  原始 skill 数：{len(mock_skills)}")
print(f"  Phase 1 过滤掉 pinned：{len(mock_skills) - len(candidates)} 条")
print(f"  进入 Phase 2 候选数：{len(candidates)}")
for s in candidates:
    print(f"    - {s['name']} (idle={s['days_idle']}d)")

=== Phase 1 预过滤 ===
  原始 skill 数：5
  Phase 1 过滤掉 pinned：1 条
  进入 Phase 2 候选数：4
    - pr-feedback-rebase (idle=5d)
    - pr-feedback-conflict-resolve (idle=8d)
    - pr-feedback-reviewer-comments (idle=12d)
    - kafka-consumer-group-debug (idle=6d)


&emsp;&emsp;`Phase 1` 输出后，进入 demo 的核心 Phase 2。

&emsp;&emsp;**B.3 Phase 2 落盘工具 + Review Prompt**——我们先定义 `manage_skill` 这个 `@tool`，它是 `review_agent` 唯一能调的工具，对应 `curator.py` 真实 `skill_manage` 的三类核心动作（`create_umbrella` / `archive` / `skip`）。同时定义 `_CURATOR_REVIEW_PROMPT_MINI`——精简移植自 `curator.py:329-358` 的真实 `CURATOR_REVIEW_PROMPT`，5 条 hard rules 在这里完整保留。

In [10]:
# B.3.1 落盘工具：manage_skill（对应 curator.py 真实 skill_manage tool）
from langchain_core.tools import tool
from pathlib import Path

CURATOR_SKILLS_DIR = Path("./curator_mvp_skills")
CURATOR_SKILLS_DIR.mkdir(parents=True, exist_ok=True)

# 模块级 list 记录 LLM 决策（对应 curator.py structured summary 的 consolidated / pruned 字段）
CURATOR_DECISIONS: list[dict] = []

@tool
def manage_skill(action: str, name: str, content: str = "", reason: str = "") -> str:
    """Curator Phase 2 review agent 的统一操作接口
       ↔ curator.py 真实 skill_manage tool（archive / patch / create）。

    Args:
        action: 三选一——"create_umbrella" / "archive" / "skip"
        name: skill 名称
        content: umbrella 合并后的 SKILL.md 完整内容（仅 create_umbrella 用）
        reason: 决策理由（教学侧用来观察 LLM 想法）
    """
    CURATOR_DECISIONS.append({"action": action, "name": name, "reason": reason})
    if action == "create_umbrella":
        umbrella_path = CURATOR_SKILLS_DIR / name
        umbrella_path.mkdir(exist_ok=True)
        (umbrella_path / "SKILL.md").write_text(content or f"# {name}\n", encoding="utf-8")
        return f"[CREATE_UMBRELLA] {name} 已落盘到 {umbrella_path}"
    elif action == "archive":
        return f"[ARCHIVE] {name} 标记归档（永远不删，可恢复）"
    elif action == "skip":
        return f"[SKIP] {name} 保留不动（无合并对象）"
    return f"[ERROR] 未知 action: {action}"


# B.3.2 精简移植自 curator.py:329-358 真实 CURATOR_REVIEW_PROMPT
_CURATOR_REVIEW_PROMPT_MINI = """你是 Hermes Curator 后台 Phase 2 LLM Review。
任务是 UMBRELLA-BUILDING consolidation——把同主题窄 skill 合并成 class-level 大伞 skill。

【候选 skill 列表（Phase 1 已过滤掉 pinned / bundled）】
{candidate_list}

【硬约束 hard rules（绝不违反）】
1. 不删除任何 skill；archive 是最大破坏性动作（archive 后文件仍在 .archive/）
2. 同主题的碎片化 skill 应该合并成 umbrella，让库保持"少而广"而不是"多而窄"
3. 不靠 'distinct trigger' 拒绝合并——pairwise 相似度不是合理门槛
4. 真无合并机会就 skip，不要硬合

【你可用的动作】
- manage_skill(action="create_umbrella", name="<class-level slug>",
               content="<umbrella SKILL.md 完整 markdown>", reason="...")
- manage_skill(action="archive", name="<old-skill>", reason="merged into umbrella")
- manage_skill(action="skip", name="<keep>", reason="no merge candidate")

【输出要求】扫一遍候选列表，识别 umbrella 合并机会，对每个候选给出最终决策；如合并 umbrella，可额外创建 umbrella + 归档原 sibling（工具调用数可能超过候选数）。
"""
print(f"[OK] manage_skill 工具和 prompt 定义完毕")

[OK] manage_skill 工具和 prompt 定义完毕


&emsp;&emsp;**B.4 Fork Curator review agent**——这是整个 demo 的灵魂。我们用 `create_agent` 工厂方法把 `review_agent` 装出来，参数链 `tools=[manage_skill]` + `middleware=[]` + `.with_config({"recursion_limit": 8})` **把 Curator Phase 2 的 4 重约束全部显式表达**（与第四节课第 2 章 Nudge demo 同构）：

In [11]:
# B.4 Fork Curator review agent —— 4 重约束的 langchain 显式表达
from langchain.agents import create_agent
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage

review_model = ChatDeepSeek(model=DEEPSEEK_MODEL)

# 与第四节 §2.8 Nudge demo 1:1 同构的 4 重约束写法
review_agent = create_agent(
    model=review_model,
    tools=[manage_skill],   # ③ 工具集隔离：review agent 只能调 manage_skill
    middleware=[],          # ④ 防递归：review agent 不再触发 Curator（也不挂任何业务 middleware）
).with_config({"recursion_limit": 8})  # ① 步数上限收紧到 8 步（对齐第四节 §2.8）
# ② quiet_mode 在 demo 里通过不打印 review_agent 内部日志体现（生产侧用 verbose=False）

# 组装候选列表传给 review_agent
candidate_list = "\n".join(
    f"- {s['name']} (idle={s['days_idle']}d, state={s['state']}): {s['summary']}"
    for s in candidates
)
prompt = _CURATOR_REVIEW_PROMPT_MINI.format(candidate_list=candidate_list)

# 执行 Phase 2 真实 LLM 调用
print("=== Phase 2 LLM Review 启动 ===")
result = review_agent.invoke({"messages": [HumanMessage(content=prompt)]})

# 展示 LLM 最终回复 + 落盘决策
final_msg = result["messages"][-1]
print(f"\n=== Phase 2 LLM 最终回复 ===")
print(f"  {str(getattr(final_msg, 'content', final_msg))[:300]}\n")

print(f"=== Phase 2 决策汇总（LLM 实际给出的 manage_skill 调用）===")
for d in CURATOR_DECISIONS:
    print(f"  [{d['action'].upper():16s}] {d['name']:35s} - {d['reason'][:60]}")

print(f"\n=== 落盘检查：./curator_mvp_skills/ 实际目录 ===")
for p in sorted(CURATOR_SKILLS_DIR.iterdir()):
    print(f"  {p.name}/")

=== Phase 2 LLM Review 启动 ===

=== Phase 2 LLM 最终回复 ===
  ## 最终决策总结

### ✅ 创建 umbrella
| Umbrella | 包含原 skill | 理由 |
|---|---|---|
| **pr-feedback-handling** | pr-feedback-rebase<br>pr-feedback-conflict-resolve<br>pr-feedback-reviewer-comments | 三者均围绕 PR reviewer 反馈后的技术操作，合并后提供完整工作流入口 |

### ✅ 归档 (merged into umbrella)
| 原 skill | 归属 |
|---|---|
| pr-feedb

=== Phase 2 决策汇总（LLM 实际给出的 manage_skill 调用）===
  [CREATE_UMBRELLA ] pr-feedback-handling                - pr-feedback-rebase, pr-feedback-conflict-resolve, pr-feedbac
  [ARCHIVE         ] pr-feedback-rebase                  - merged into pr-feedback-handling umbrella — rebase 操作作为子操作之一
  [ARCHIVE         ] pr-feedback-conflict-resolve        - merged into pr-feedback-handling umbrella — merge conflict r
  [ARCHIVE         ] pr-feedback-reviewer-comments       - merged into pr-feedback-handling umbrella — reviewer comment
  [SKIP            ] kafka-consumer-group-debug          - 独立中间件调试技能，与 PR 反馈处理组无主题关联。无其他 Kafka 相关 skill 待合并，保留原样。

&emsp;&emsp;运行这段代码会看到 `review_agent` 调用 `DeepSeek` 模型，按 prompt 的指引扫一遍候选列表，识别 `pr-feedback-*` 三条作为 `umbrella` 候选，调用 `manage_skill(action="create_umbrella", name="pr-feedback", ...)` 创建合并 skill，再 `archive` 掉三条原 sibling，最后对 `kafka-consumer-group-debug` 调用 `skip`（无合并对象）。**注意 `critical-prod-runbook` 从未出现在 `CURATOR_DECISIONS` 里**——它在 Phase 1 就被 `pinned` 过滤掉了，这是 5 条护栏中 pinned 跳过的真实效果。

&emsp;&emsp;**步骤三：观察点 + Tier 2 验证**

&emsp;&emsp;关键观察点有四个：① **umbrella 合并真发生**——`pr-feedback-*` 三条被合并成 `pr-feedback` 大伞，`./curator_mvp_skills/pr-feedback/SKILL.md` 真落盘；② **archive 不是 delete**——LLM 调的是 `manage_skill(action="archive", ...)` 而不是删除工具（hard rule 1 落地）；③ **pinned 完全不进决策**——`critical-prod-runbook` 在整个 `CURATOR_DECISIONS` 列表里完全找不到（Phase 1 已过滤）；④ **`recursion_limit=8` 真生效**——`review_agent` 在合并 + archive + skip 总共调 5 次工具后自然停止，不会无限循环。

&emsp;&emsp;**Tier 2 验证（断言）**——对 LangChain demo 加一组 assert 确保 Curator 五条护栏在 LLM 决策侧真正触发：

In [12]:
# Tier 2 端到端验证：assert Curator 双 Phase 闭环的关键护栏全部触发
umbrellas = [d for d in CURATOR_DECISIONS if d["action"] == "create_umbrella"]
archived  = [d for d in CURATOR_DECISIONS if d["action"] == "archive"]
skipped   = [d for d in CURATOR_DECISIONS if d["action"] == "skip"]

# 护栏 1：umbrella 合并真发生（至少 1 个 pr-feedback umbrella）
assert len(umbrellas) >= 1, f"至少应该创建 1 个 umbrella（pr-feedback 系列），实际 {len(umbrellas)}"
assert any("pr" in u["name"].lower() for u in umbrellas), \
    f"umbrella 名称应该体现 pr 主题，实际 {[u['name'] for u in umbrellas]}"

# 护栏 2：archive 至少 2 条（pr-feedback-* 中至少 2 条被合并归档）
assert len(archived) >= 2, f"至少 2 条窄 skill 被合并归档，实际 {len(archived)}"

# 护栏 3：pinned skill 不出现在决策列表
all_decided_names = [d["name"] for d in CURATOR_DECISIONS]
assert "critical-prod-runbook" not in all_decided_names, \
    "pinned skill 不应该出现在 Curator 决策列表（Phase 1 已过滤）"

# 护栏 4：umbrella 真落盘到磁盘
assert (CURATOR_SKILLS_DIR / umbrellas[0]["name"] / "SKILL.md").exists(), \
    "umbrella 应该落盘到 ./curator_mvp_skills/<name>/SKILL.md"

print("\n[OK] Phase 2 LLM Review 关键护栏全部触发：")
print(f"  [通过] umbrella 合并真发生：{len(umbrellas)} 个 umbrella（{umbrellas[0]['name']}）")
print(f"  [通过] archive 不删原则：{len(archived)} 条窄 skill 归档")
print(f"  [通过] pinned 跳过：critical-prod-runbook 完全不进决策")
print(f"  [通过] umbrella 真落盘：SKILL.md 文件存在")


[OK] Phase 2 LLM Review 关键护栏全部触发：
  [通过] umbrella 合并真发生：1 个 umbrella（pr-feedback-handling）
  [通过] archive 不删原则：3 条窄 skill 归档
  [通过] pinned 跳过：critical-prod-runbook 完全不进决策
  [通过] umbrella 真落盘：SKILL.md 文件存在


&emsp;&emsp;这段 assert 全部通过 → 说明已经完整理解 `Curator` 双 Phase 闭环。亲手让 LLM 跑过一次 umbrella 合并之后，我们才能真正回答"为什么 `Curator` 不是去重器、不是删除器，而是聚合器"——因为它的目标是让 skill 库**少而广**而不是**多而窄**。下一步进入第 2 章——`Curator` 第 5 条护栏（"只动 agent-created"）背后的工程实现，`ContextVar` 血统隔离。

&emsp;&emsp;> **【学完本节你已经掌握】**：① `Curator` 是 idle-triggered 双 Phase（30/90 天状态机 + LLM umbrella 合并）；② 五条护栏的核心是"永远不删 + 克制 + 可回滚"；③ 跑过 LangChain 模拟版亲眼看到 LLM 真实给出的合并决策。下一章我们打开 `Curator` 第 5 条护栏（"只动 agent-created"）的工程实现——`ContextVar` 血统隔离。

&emsp;&emsp;在进入第 2 章前再做一次"对照第四节课默认链路"的差异锁定。第四节课第 2 章 看到的 `Nudge` 是 skill 的"生产端"——计数器到了 fork agent 写一条 skill 落盘。今天看到的 `Curator` 是 skill 的"质检端"——按周扫一次把 skill 在三态间迁移、把同质 skill 合并成 umbrella。两个机制合在一起就是 `Hermes` 完整的"自学习闭环"——使用 → `Nudge` 计数 → fork → 写 skill → idle → `Curator` 整理 → 质量可控 → 更好的 skill → 回到使用。这个闭环我们看完整本课会在第 3.1 节看到完整的飞轮图。但**自学习闭环的真实边界在哪里**——这个问题要等到第 3 章才能完整回答。第 2 章我们先解决一个更直接的工程问题：用户手写的 skill 怎么避免被这个闭环误伤。

## <center>第2章：Skill 血统隔离——5 月新增机制</center>

&emsp;&emsp;第 1 章把 `Curator` 的整套机制走完了——五条护栏、双 Phase 流程、状态机分档。但还有一个地方值得追问：**`Curator` 是怎么知道一条 skill 是 agent 写的、还是用户手写的？**

&emsp;&emsp;如果用户上周精心改好了一条 skill，但这两周没开 `Hermes`，下次 idle 触发时这条 skill 会不会被 `Curator` 当成"agent-created"一起整理掉？这正是 Reddit 上被顶到 107 票的痛点。

&emsp;&emsp;`Hermes` 团队在 5 月的两个修补提交里给出了答案：一个叫 `skill_provenance` 的 78 行模块。本章打开这段代码，看 `Hermes` 是怎么用最轻量的方式给 `Curator` 装上第五条护栏——**只动 agent 自己写的 skill，绝不碰用户手写的**。

### 2.1 问题引入：Reddit 高赞吐槽与业界对标

&emsp;&emsp;先看 Reddit 上把这个问题顶上 107 票最高位的原话，原文是英文的——保留它的原始形态比转述更有冲击力：

&emsp;&emsp;> &emsp;"So then you go and manually edit the skill. Fix it. And it does a good job.. GUESS WHAT. It's self improving. It will overwrite your edits."
> &emsp;（中文："你动手把这条 skill 改对了，agent 也做得不错。然后呢？这玩意儿是『自我改进』的——它会把你的修改覆盖回去。"）

&emsp;&emsp;翻译过来：用户发现 agent 写的某条 skill 不太合用，于是动手把那个 skill 改对了——下一个 nudge 周期一来，agent 自动把用户的改动覆盖掉重新写回它自己版本。这是任何"自学习闭环"系统都绕不开的问题——agent 写的 skill 和用户手写的 skill 混在同一目录，`Curator` 整理时分不清哪些可以动、哪些是用户的私产。`Hermes` 在 5 月的两个修补提交里把这个问题修了：**第一个提交**引入 `tools/skill_provenance.py`（声明 `ContextVar` API），**第二个提交**改写 `mark_agent_created()` 调用条件——5 月前所有 `skill_manage(create)` 都标记 `created_by="agent"`，5 月后只有当**当前执行上下文是 background_review**时才标记。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141450442.png" width=60%></div>

&emsp;&emsp;**业界对标——不是每个 agent 产品都会撞上这个痛点**。痛点出现的精确条件是 ① agent 完全无 user 干预的自动写 + ② 后台还会自动 review 已写过的文件。两个条件同时满足才会出现"用户手改被自动覆盖"。下表把主流产品现状和工程路径合并展示——前四行说明它们为什么没撞上，最后一行是本课主角 `Hermes` 选的路径：

<p align="center">表 2-0 主流 agent 产品在"双后台"两轴的位置 + 各自工程路径（2026-05 当下）</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 产品 / 形态 | ① 自动写 | ② 自动 review | 撞痛点 | 工程路径 |
| :---: | :---: | :---: | :---: | :---: |
| `Cursor` / `Cline` / `Aider` 主编辑 | 否（diff+accept） | 否 | 不撞 | 路径0：天然 user 审批 |
| `Cursor Background Agents` | 部分（远端分支异步） | 否 | 不撞 | **路径3**：git 分支 + 人工 PR merge |
| `Cursor Memory` / `Claude Code` Auto memory | 弱（多数路径需批准） | 否（写完不再 review） | 不撞 | **路径4**：approval queue |
| `Claude Code` `#` 记忆 | 否（user 输入触发） | 否 | 不撞 | 路径0：天然 user 审批 |
| **`Hermes` Nudge + Curator** | **是** | **是** | ⚠️ **必撞** | **路径5**：`ContextVar` 隐形血统标签（**本课**） |

&emsp;&emsp;**表 2-0 揭示一个反直觉事实——主流 agent 产品大多没撞上这个痛点（前四行 ① 或 ② 条件不满足），唯有 `Hermes` 的双后台设计同时满足两条——必然撞上。** **更通用的工程原则**：凡 agent 生成的持久化产物未来可能被读取、整理、覆盖，都应有 provenance（血统）标签——双后台只是这条原则下最高风险的形态。无论选哪条路径（git PR / approval queue / 隐形标签），背后都绕不开"**打标 → 保真 → 决策**"三段闭环（即使 git PR 隔离，本质也是用 commit 元数据当 provenance）。下面看 `Hermes` 选的路径 5 的代码实现。

### 2.2 为什么需要"两步走"——ContextVar 不跨线程传播

&emsp;&emsp;> 📚 **前置基础不熟可先看附属课件**：若对 `threading.Thread` vs `asyncio.create_task`、`ContextVar` 跨线程不传播、`copy_context()` 标准解法这些概念不熟，建议先打开同目录《**并发与ContextVar.ipynb**》补完 9 节并发基础（约 25 分钟），再回来读本章——下面只讲结论，不再展开并发原理。

&emsp;&emsp;`Hermes` 用的是 `threading.Thread`（不是 `asyncio.create_task`），所以纯靠 `ContextVar` 自动传播行不通——新线程的 context 是空的，父线程 `set` 的值在子线程里读到的还是默认值。

&emsp;&emsp;Python 标准解法是 `copy_context() + ctx.run(func)`，但 `Hermes` 选了更轻量的"**两步走**"：

① **父线程染色实例属性**（`review_agent._memory_write_origin = "background_review"`，普通属性可跨线程读取）；

② **子线程入口绑定 ContextVar**（在 `run_conversation()` 顶部调用 `set_current_write_origin(self._memory_write_origin)`，把属性值"搬"进本线程的 `ContextVar`）。两步合起来等价于 `copy_context()`，但只传播一个业务值，更轻量。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141443031.png" width=60%></div>

&emsp;&emsp;下面看 `tools/skill_provenance.py` 怎么把"两步走"包装成 4 个 `API`，让所有写入点都用同一套标准接口。

### 2.3 ContextVar 机制原理：①打标 + ②保真两段闭环

&emsp;&emsp;`tools/skill_provenance.py` 总共 78 行，核心只有 4 个函数和 1 个全局 `ContextVar`。本节看 provenance 闭环的前两段——**② 保真**（怎么把"染色信号"搬到子线程的 `ContextVar`）+ **① 打标**（怎么在 skill 写入时按 `ContextVar` 打 `created_by` 标签）。第三段 **③ 决策**（Curator 怎么按标签过滤）在下一节 2.4。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513144256956.png" width=60%></div>

&emsp;&emsp;先看 `ContextVar` 声明 + ② 保真 API（节选 `skill_provenance.py:37-79`，4 个 API 中的两个，另两个 `reset / get` 接口对称略）：

In [41]:
# tools/skill_provenance.py:37-79 节选 —— ContextVar 声明 + ②保真 API
import contextvars

# default="foreground" 让"没设置时默认认为是用户前台调用"——5月前漏标的 skill 都是这个状态
_write_origin: contextvars.ContextVar[str] = contextvars.ContextVar(
    "skill_write_origin", default="foreground"
)
BACKGROUND_REVIEW = "background_review"  # 字符串常量化，避免拼写漂移

def set_current_write_origin(origin: str):
    """子线程入口调用：把父线程传来的 origin 值绑到本线程的 ContextVar。返回 token 用于后续 reset。"""
    return _write_origin.set(origin or "foreground")

def is_background_review() -> bool:
    """写入点调用：判断当前执行上下文是否为 background_review。"""
    return _write_origin.get() == BACKGROUND_REVIEW

&emsp;&emsp;`Hermes` 在 `run_agent.py:11491` 附近（`run_conversation` 顶部）调用 `set_current_write_origin(self._memory_write_origin)` 这**一行**——把父线程通过实例属性传来的 `"background_review"` 值绑到本线程 `_write_origin`。**这是 ② 保真的全部工程量。**

&emsp;&emsp;再看 ① 打标。`skill_manager_tool.py:792-796`——所有 `skill_manage(create)` 操作都必经这个判断：

In [42]:
# tools/skill_manager_tool.py:792-796 —— ①打标：按 ContextVar 状态决定是否标 created_by
from tools.skill_usage import mark_agent_created
from tools.skill_provenance import is_background_review

if is_background_review():
    mark_agent_created(name)

&emsp;&emsp;这 5 行是 ① 打标的全部代码——`is_background_review()` 返回 `True` 才标 `created_by="agent"`，foreground agent 的写入不染色，`created_by` 保持空。**设计的优雅在于：跨层传递的"执行上下文信息"，用 Python 标准库零成本实现，不需要显式传参、不需要全局 dict、不需要新的数据库字段。**

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-1 两路写入来源 × ContextVar 状态 × Curator 权限对照</font></p>
<div class="center">

| 写入来源 | `_write_origin` 值 | `mark_agent_created` 是否触发 | `created_by` 字段 | `Curator` 权限 |
|---|---|---|---|---|
| 用户让 foreground agent 写（"帮我写条 skill"）| `"foreground"`（默认） | **[否]** | 空 / `None` | **不动**——Phase1 跳过、Phase2 prompt 限制 |
| `Nudge` fork review agent 自动写 | `"background_review"` | **[是]** | `"agent"` | **可整理**——状态机流转 |

</div>


&emsp;&emsp;表 2-1 把"两路来源 × 三字段 × 权限"压成一张对照表。**最右一列的权限完全由最左一列的上下文自动决定**，中间没有任何手动判断——这正是 `Hermes` 选 `ContextVar` 的原因，而不是"给 `skill_manage(create)` 显式加一个 `creator: str` 参数"（后者要在每个调用点都改代码）。

&emsp;&emsp;78 行声明 + 5 行调用条件，解决了"用户 skill 不被覆盖"这个工程问题。下一节看 ③ 决策——`Curator` 怎么消费 `created_by` 标签，让闭环真正闭上。

### 2.4 ③ 决策：Curator 双闸消费 created_by 标签

&emsp;&emsp;`ContextVar` 染色只是"打标"——另一半是 `Curator` 在执行整理时**真的去看这个染色字段**。前面 1.3 / 1.4 节讲过 `Curator` 双 Phase 流程，这里只看血统检查的两处具体落地点。

&emsp;&emsp;**Phase 1 自动 Transition——③ 决策的真源码**。`apply_automatic_transitions` 在 `agent/curator.py:255-271` 扫描所有 skill 时，**第一行判断**就是按 `created_by` 字段过滤：

In [43]:
# agent/curator.py:269 真源码 —— ③ 决策：从生成端就只拿 agent-created skill
def apply_automatic_transitions(now=None):
    # _u.agent_created_report() 内部已按 created_by="agent" 过滤
    # —— 用户 skill 根本不进入这个 for 循环，等价于"if created_by != 'agent': continue"
    for row in _u.agent_created_report():
        counts["checked"] += 1
        if row.get("pinned"):
            continue
        # 下面才是 active → stale → archived 的状态机迁移逻辑
        ...

&emsp;&emsp;注意工程姿态：`Hermes` 没写 `if skill.created_by != "agent": continue` 这种显式过滤，而是把过滤**前置到候选生成函数** `_u.agent_created_report()` 里——用户 skill 根本不进入循环。这比显式出口阀更安全：① 打标 + ② 保真的所有工作，让这个函数的返回列表天然就不含用户 skill。

&emsp;&emsp;**Phase 2 LLM Review**：`CURATOR_REVIEW_PROMPT` 第 1 条 hard rule 是 `"DO NOT touch bundled or hub-installed skills. The candidate list below is already filtered to agent-created skills only."`——明示 Phase 2 LLM 拿到的 candidate list 已经被过滤为 agent-created。**Phase 1 候选构建在源头过滤用户 skill，Phase 2 prompt 约束 LLM 不越界动 bundled/hub**——一道闸守源头 + 一道闸守边界，分工清楚。

&emsp;&emsp;**为什么必须看两个提交**：5 月前，`mark_agent_created()` 在所有 `skill_manage(create)` 时都被调用（包括 foreground 用户主动让 agent 写的），导致用户 skill 也被标 `created_by="agent"`。5 月后只在 `is_background_review()` 为真时才标——这就是为什么"5 月后 foreground 用户 skill 不进入 Curator 候选"，真因不是 `Curator` 改了，而是**写入端的标记逻辑改了**。两个提交缺一不可，看任何"已修复"的 PR 都要看完整修补链。

&emsp;&emsp;讲到这里已经把 `ContextVar` 血统机制的完整逻辑链走完——`_write_origin` 染色 → `is_background_review()` 判断 → `mark_agent_created()` 选择性标记 → `Curator` 双 Phase 都按 `created_by` 过滤。下面通过 Demo 2 用**教学版 stub** 同构复刻 `Hermes` 主仓的 `skill_manage(create)` + `skill_provenance` API（与 `skill_manager_tool.py:792-796` 5 行打标逻辑完全同构），亲眼看到 `created_by` 字段在两路来源下的实际值。

#### 2.4.1 Demo 2：Skill 血统验证（5 min）

&emsp;&emsp;Demo 2 把表 2-1 的"理论对照"变成"亲眼看到的真实输出"。教学版 stub 同构复刻 + 真源码对位可选——本地无 HermesAgent 源码也能完整跑。一段代码完成三件事：① 两路写入（foreground 默认 + background_review 染色后）② 读取 `_usage_db` 的 `created_by` 字段 ③ Tier 2 断言血统契约。

In [13]:
# Demo 2：Hermes 血统机制端到端验证（教学版同构 stub）
# 真源码对位见 tools/skill_provenance.py:37-79 + tools/skill_manager_tool.py:792-796
import contextvars
from typing import Dict, Any

# === 教学版 stub：复刻 skill_provenance + skill_usage 核心 API ===
_write_origin = contextvars.ContextVar("skill_write_origin", default="foreground")
BACKGROUND_REVIEW = "background_review"
_usage_db: Dict[str, Dict[str, Any]] = {}  # 内存模拟 ~/.hermes/skills/usage.json


# 判断当前是 background_review 吗？
def is_background_review() -> bool:
    return _write_origin.get() == BACKGROUND_REVIEW


# 在数据库里给这个 skill 打上 created_by="agent"
def mark_agent_created(name: str) -> None:
    _usage_db.setdefault(name, {})["created_by"] = "agent"


# 写 skill 的入口——5 行核心逻辑在这里
def skill_manage(action: str, name: str, content: str = "") -> str:
    """同构复刻 tools/skill_manager_tool.py:792-796 核心打标逻辑。"""
    if action == "create":
        _usage_db.setdefault(name, {})["content"] = content   # 先把内容存进去
        if is_background_review():     # ← 读标牌，判断调用者身份，来自真源码 :792-796
            mark_agent_created(name)   # ← 只有 background 才打标
        return f"created: {name}"
    raise ValueError(action)

# === 路径 1：foreground 用户让 agent 写 skill（ContextVar 默认值，不染色）===
skill_manage("create", "user-skill", "# User Skill\n")

# === 路径 2：Nudge fork review agent 染色 _write_origin 后写 skill ===
token = _write_origin.set(BACKGROUND_REVIEW)       # ① 染色：把标牌换成 background_review
try:
    skill_manage("create", "agent-skill", "# Agent Skill\n")   # ② 写 skill，内部读到 background → 打标
finally:
    _write_origin.reset(token)  # ③ 还原：标牌恢复原状，不影响后续代码

# === Tier 2 断言：血统契约必须成立 ===
assert "created_by" not in _usage_db["user-skill"], "user-skill 不应被标 agent-created"
assert _usage_db["agent-skill"]["created_by"] == "agent", "agent-skill 必须标 created_by='agent'"

print(f"user-skill   created_by = {_usage_db['user-skill'].get('created_by')!r:8} → Curator 不动")
print(f"agent-skill  created_by = {_usage_db['agent-skill'].get('created_by')!r:8} → Curator 可整理")
print("\n[OK] 两路血统契约成立——这就是 Curator 第五条护栏'只动 agent-created'的工程底座")

user-skill   created_by = None     → Curator 不动
agent-skill  created_by = 'agent'  → Curator 可整理

[OK] 两路血统契约成立——这就是 Curator 第五条护栏'只动 agent-created'的工程底座


In [14]:
cv = contextvars.ContextVar("x", default="A")
print(f"初始值:            cv.get() = {cv.get()!r}")

token = cv.set("B")
print(f"set('B') 之后:     cv.get() = {cv.get()!r}")
print(f"token.old_value  = {token.old_value!r}  ← MISSING 表示之前只有 default 没被显式 set 过")
print(f"token.var        = {token.var}")

cv.reset(token)
print(f"reset(token) 之后: cv.get() = {cv.get()!r}  ← 恢复到 A")

初始值:            cv.get() = 'A'
set('B') 之后:     cv.get() = 'B'
token.old_value  = <Token.MISSING>  ← MISSING 表示之前只有 default 没被显式 set 过
token.var        = <ContextVar name='x' default='A' at 0x1303b3ab0>
reset(token) 之后: cv.get() = 'A'  ← 恢复到 A


&emsp;&emsp;运行后输出 `user-skill created_by = None`、`agent-skill created_by = 'agent'`，两条断言通过——这就是 ① 打标 + ② 保真 + ③ 决策三段闭环在最小代码下的完整演示。同一个 `skill_manage(create)` API 在 `ContextVar` 默认值下走"用户路径"、在 fork 染色后走"agent 路径"，两路产物在 `created_by` 字段上完全分开——`Curator` 后续整理时所有判断都从这一个字段出发。

&emsp;&emsp;> **【学完本章你已经掌握】**：① **痛点边界**——双后台（自动写+自动 review）是 provenance 必备的最高风险形态，主流 agent 产品大多通过 user 审批 / git PR / approval queue 规避，`Hermes` 选了"隐形血统标签"路径；② **两步走**——父线程染色实例属性 + 子线程入口绑定 `ContextVar`，等价于纯 asyncio 的 `copy_context()` 但只传一个业务值；③ **三段闭环**——`_write_origin` 染色（②保真）→ 5 行 `if is_background_review(): mark_agent_created()`（①打标）→ `_u.agent_created_report()` 源头过滤（③决策）；④ **5 月双提交合力**——引入 `ContextVar` 染色 + 改写 `mark_agent_created` 调用条件（只在 `is_background_review()` 为真时才标），缺一不可。

&emsp;&emsp;进入第 3 章前再做一次"对照第四节课默认链路"的差异锁定。第四节课第 2 章看到的 `Nudge` 是 skill 的"生产端"——计数器到了 fork agent 写 skill；今天第 1 章的 `Curator` 是 skill 的"质检端"——按 idle 触发整理；第 2 章的 `ContextVar` 血统隔离是质检端的"准入门"——决定哪些 skill 才能进入质检流程。三者合在一起就是 `Hermes` 完整的"自学习闭环 + 边界保护"。但**这个闭环的真实进化能力到底有多大**——它能不能让模型本身变得更聪明？skill 内容能不能自动越写越好？讲义里厂商宣称的"越用越强 + 2-3 倍效率提升"是不是真的？这些问题就是第 3 章——本课最硬核的板块——要逐条用源码回答的。

## <center>第3章：自进化的诚实边界</center>

&emsp;&emsp;前两章把 `Hermes` 自学习闭环的工程机制走完了——`Nudge` 生产 skill、`Curator` 质检、`ContextVar` 保护用户 skill 不被覆盖。但还有一个核心问题没有回答：**到底是什么在"进化"？模型变聪明了吗？skill 怎么从一句话变成精确指令？**

&emsp;&emsp;这一章聚焦一件事：把 `Hermes` 自进化飞轮里**真正干活的算法 `GEPA`** 讲清楚，然后用 Demo 3 亲手跑一次，看见 prompt 自动进化的真实形态。

&emsp;&emsp;为什么值得花一整章讲它？因为跑完 Demo 3 的 30 行代码，你会亲眼看到一句 prompt 从一个字 `"Reply."` 进化成一段六条规则的英文说明，accuracy 从 50% 升到 100%。**模型没变，变的是 prompt**——这就是"越用越强"的真实作用域。

### 3.1 飞轮图：Nudge × Curator 自进化闭环

&emsp;&emsp;现在把三章的内容拼成一张完整的"自进化飞轮"图。`Nudge` 部分第四节课已经讲过，这里只需要记住那条链路：计数器到 10 → fork review agent → 反思最近 10 次工具调用 → 写 skill 落盘。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513141449650.png" width=70%></div>

&emsp;&emsp;这张图的关键不是它转得多漂亮，而是**它的边界在哪**。两条边界要记住：

- **物理边界**（图中红虚线圈）：按当前源码，foreground 用户 skill 不进入 `Curator` 候选

- **认知边界**（图外圈底部）：飞轮转的是 skill 内容、`MEMORY.md` 事实、tool 调用模式，**不会让模型变聪明**

&emsp;&emsp;§3.2 把这条认知边界讲清楚，§3.3 进入 `GEPA` 本身。

### 3.2 自进化边界：进化的是 prompt 注入质量

&emsp;&emsp;先把这个问题说清楚：`Hermes` 的"越用越强"**真有所指，但所指的对象不是模型本身**。`Hermes` 进化的是 **prompt 注入的质量**——模型能力天花板不变，但每次启动时注入的 skill 越来越精准、越来越全面。下面这张对照表把"能进化的"和"不能进化的"并排列出：

<p align="center">表 3-1 能进化 vs 不能进化（自学习闭环的真实作用域）</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 能进化的（机制层） | 不能进化的（模型层） |
| :---: | :---: |
| `skill` 内容（Markdown 文本，`Curator` umbrella 合并 + LLM review 改写） | LLM 模型参数（权重不变，没有 fine-tune（微调）） |
| `MEMORY.md` 中的事实（用户偏好、项目背景、历史决策被 review agent 累积写入） | 模型的推理能力（同一个 GPT-4o 就是同一个 GPT-4o） |
| `USER.md` 中的用户画像（`Honcho` plugin 启用时辩证更新） | 模型的知识边界（训练截止日期不会因为 `Hermes` 多用而前移） |
| tool 调用模式（哪些 tool 在哪种场景下用得好，写进 skill 形成模式库） | 通用世界知识（模型不会因为用 `Hermes` 多就突然懂量子物理） |

&emsp;&emsp;表 3-1 左列每一行都是"`Hermes` 真能做到"的事情；右列是"任何不带 fine-tune 的 agent 都做不到"的边界。**问题来了——左列里 `skill` Markdown 内容是怎么"越合越精"的？** 答案是：`Hermes` 厂商在独立子项目 `hermes-agent-self-evolution` 里用 `GEPA` 算法做 skill 文本演进——读 baseline skill 在历史 task 上的失败案例 → 反思变异生成更精确的 skill markdown 候选 → 帕累托前沿筛选保留。下一节 §3.3 我们不展开这个独立子项目，而是**直接用 `DSPy` 内置的 `dspy.GEPA` 把同款 `GEPA` 算法本身讲清楚**——理解算法核心后，回头看 Hermes 子项目就能秒懂。

&emsp;&emsp;> 📌 **本讲新术语 30 秒速查**（§3.3 / §3.4 即将密集使用）：
> &emsp;`DSPy` = Stanford NLP 出的声明式 LLM 编程框架，把 prompt / module / signature 用 Python 类抽象出来。
> &emsp;`GEPA` = Genetic-Pareto，DSPy 内置的 prompt 自动优化器，核心机制"反思变异 + 帕累托前沿"。
> &emsp;`Pareto`（帕累托前沿）= 多目标优化保留的候选池——只要在某些验证样本上得分最高就保留，覆盖度抽样进下轮。
> &emsp;`reflection_lm` = GEPA 用来读错样本并改写 instruction 的反思模型，可与 baseline LM 不同。
> &emsp;`litellm` = DSPy 底层使用的跨模型路由库，用 `provider/model` 前缀统一调 OpenAI/DeepSeek/Anthropic 等。
> &emsp;`signature.instructions` = DSPy 编译 `Signature` 子类 docstring 后的运行时字段，GEPA 真正改写的对象。

### 3.3 GEPA 是什么 · 一句话三问

&emsp;&emsp;`Hermes` 把 prompt 进化背后的算法叫 `GEPA Skill Evolution`，放在独立子项目 `hermes-agent-self-evolution` 里（不在主仓默认链路）。本节不展开这个子项目，而是**直接用 `DSPy` 内置的 `dspy.GEPA` 把算法本身讲清楚**——任何人 `pip install dspy-ai` 后就能直接用。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513144256936.png" width=60%></div>

&emsp;&emsp;> 📌 **重要划界 · Demo 3 跑的不是 Hermes 子项目本身**：两者**机制完全同源**（反思变异 + 帕累托前沿筛选），**优化对象不同**——

① `hermes-agent-self-evolution` 子项目的 `GEPA Skill Evolution` 改的是 `~/.hermes/skills/*/SKILL.md` 这份 Markdown **文件文本**；

② 本课 Demo 3 的 `dspy.GEPA` 改的是 DSPy `Signature` 编译后的 `signature.instructions` **内存字段**（不是文件）。核心算法 1:1 一致，所以理解了 Demo 3 就理解了 GEPA 本身；落地形态差异（文件 vs 字段）不影响算法理解。这也是后面 §3.5 表 3-2 那句"`GEPA` 改 `instructions` 不改 `skill_text`"的精确含义——指**Demo 3 这个具体跑法**只动 instructions 字段，不写回 Hermes skill 文件。

github地址：https://github.com/NousResearch/hermes-agent-self-evolution/tree/main

&emsp;&emsp;下面用三问把 `GEPA` 这个概念交底——三个问题，30 秒看完。

&emsp;&emsp;**问 1：GEPA 是什么？** —— `GEPA` 全称 `Genetic-Pareto`（遗传算法 + 帕累托前沿），是 `Berkeley/Stanford/MIT 联合团队` 团队提出的 prompt 自动优化器（论文 `ICLR 2026 Oral`），现已合入 `DSPy` 框架作为 `dspy.GEPA` 直接调用。它跟 `MIPROv2`（DSPy 内置的另一款 prompt 优化器，走"贝叶斯搜索 + 少样本自举"路线）是同一个生态位上的两套思路——一个偏反思派、一个偏稳健派。

&emsp;&emsp;**问 2：GEPA 怎么做？** —— 跟传统优化器最大的不同是"反思变异"四个字。一句话讲完它的核心动作链：

① 用一份训练样本跑一遍当前 instruction（指令文本）→ 

② 看哪几条样本答错 → 

③ 让一个"反思 LLM"（通常是比 baseline 更强的模型——Demo 3 里 baseline 用 `deepseek-chat`、反思 LLM 也用 `deepseek-chat`（DeepSeek reasoner 会输出 `<think>` token 干扰 DSPy 解析，所以反思 LLM 不切 reasoner））读这几条错样本，思考"上次错在哪、这次该怎么改" → 

④ 生成 N 个改写后的 instruction 候选 → 

⑤ 把候选放进帕累托前沿（按验证样本表现维护的候选池——候选只要在某些验证样本上得分最高就被保留下来，再按覆盖度抽样进入下一轮）→ 

⑥ 下一轮再迭代。整个过程**不调整模型权重，只重写 instruction 文本**——这就印证了 §3.2 那条"进化的是 prompt 注入质量，不是模型能力"的核心论断。

&emsp;&emsp;**问 3：GEPA 为什么有效？** —— 一句话：它把"调 prompt"这件事从"工程师拍脑袋手写"变成"让大模型读自己上次错在哪、自动反思后改写"。论文官方数字：在 `MATH benchmark`（数学推理基准测试）上 baseline `70%→evolved 92%`；本课 §3.4 Demo 3 在"JSON 输出"任务上讲师 2026-05 在 `harness` conda 环境用 `deepseek-chat` 实测 `50%→100%`（+50%）（两组数字任务难度完全不同不可互比，方向相同——都是"反思变异显著拉升 accuracy"）。"反思变异"是 GEPA 区别于其他 prompt 优化器的唯一关键——记住这一点，下面 §3.4 Demo 3 的 instruction diff 才有理解锚点。

&emsp;&emsp;> **30 秒自测（进入 Demo 3 前确认）**（进入 Demo 3 前）：
> &emsp;① `GEPA` 改的是什么、不改什么？
> &emsp;② `reflection_lm` 做什么、能不能跟 baseline LM 用同一个模型？
> &emsp;③ 帕累托前沿保留候选的依据是什么？
> &emsp;答得上 = 直接进 Demo 3；答不上 = 回 §3.3 三问扫一眼。

### 3.4 Demo 3 · 30 行 dspy.GEPA 真跑

&emsp;&emsp;讲完 GEPA 三问，下面我们**亲手跑一次 `GEPA`**——让我们看见 `GEPA` 把一句极弱 instruction 自动进化成一段多约束英文规则文本的真实形态。Demo 3 任务设计：让模型把任意请求转成严格的 JSON 输出。

&emsp;&emsp;**Demo 3 任务设计**：让模型把任意请求转成严格的 JSON 输出。Baseline 用极弱 instruction `"Reply."`（一个字）让 GEPA 有反思空间——这种 baseline 下 `deepseek-chat` 容易把商品列表输出成 markdown 编号格式（譬如"1. 商品1: {id...} 2. 商品2: {...}"），`GEPA` 反思后给出一段精确的英文规则段落（覆盖"reasoning + output 双字段输出"/ "use double quotes for all strings" / "never wrapped in a Markdown code block" / 嵌套对象/数组对象的字段规范等约束）。选这种"弱 baseline + JSON 输出"对比的四个理由：① 失败模式直观（学员一眼看出 markdown 编号格式是错）；② 反思 NL 内容自然（"模型有时输出 markdown 列表，要禁止"）；③ Score 提升肉眼可见（50% → 100%，绝对提升 50 个百分点）；④ 5-10 个样本即可冷启动，避免数据准备成本。

&emsp;&emsp;**Cell 1：安装 dspy + 配置 LM**

&emsp;&emsp;先装 `dspy-ai`（独立包，不依赖 `Hermes` 主仓）+ 配置一个 LM 做 baseline。这里用 `deepseek-chat` 是因为它在国内访问稳定、性价比高（约 ¥0.001/1K tokens）、且 DeepSeek 提供 OpenAI-compatible 接口协议，可被 `dspy.LM` 通过 `litellm`（DSPy 底层使用的跨模型路由库，用 `provider/model` 前缀统一调多家模型）原生 `deepseek/` 前缀无缝接入。如果你用 `Claude` 或 `Qwen`，把 `dspy.LM(...)` 那行改成对应模型名即可。

In [15]:
# ── Cell 1：安装 dspy-ai + 配置 LM ───────────────────────────────
# 锁定 3.x 版本区间，避免 GEPA API 漂移（2026-05 实测 3.2.1 可用）
import dspy
import json
import os
from pathlib import Path

# DEMO3_LM 在 §0.4 env cell 中已定义（默认 "deepseek/deepseek-chat"）
# 如果你从这里单独开始跑，确保 .env 已加载：from dotenv import load_dotenv; load_dotenv()
dspy.configure(lm=dspy.LM(DEMO3_LM))

print("[OK] dspy 已配置，准备定义 SkillModule")

[OK] dspy 已配置，准备定义 SkillModule


&emsp;&emsp;Cell 1 跑通后应该看到 `[OK]` 输出。如果你不打算真跑（课件已附带 cached 结果），这个 cell 跑不跑都不影响后面——`gepa_cache.json` 已经把 baseline / evolved 结果都存好了。

&emsp;&emsp;**Cell 2：定义 SkillModule（DSPy 标准结构）**

&emsp;&emsp;这个 module 是 DSPy 标准结构——`TaskSig` 这个 `dspy.Signature` 子类的 **docstring**（`"""Reply."""` 三引号文本）在 DSPy 编译时被搬到 `signature.instructions` 字段，作为 baseline instruction。`predictor`（DSPy 中执行具体推理任务的对象）是核心 `ChainOfThought` predictor。`GEPA` 反思变异时改写的是**编译产物 `signature.instructions`**（不是 Python 源码里的 docstring 字面量字符串）——DSPy 3.x 真实访问路径是 `predictor.predict.signature.instructions`，因为 `ChainOfThought` 内部包了一层 `Predict`。这印证了 §3.3 问 2 动作链 ⑤"在帕累托前沿里保留得分最高的 instruction 候选"——保留的是**运行时字段值**，不是源码 docstring。

In [16]:
# ── Cell 2：定义 SkillModule（DSPy 标准结构）────────────────────────

class TaskSig(dspy.Signature):
    """Reply."""
    # ↑ 这就是 baseline instruction（极弱，刻意留给 GEPA 反思空间）
    # DSPy 编译时会把这个 docstring 搬到 signature.instructions 字段
    # GEPA 优化的正是这个字段——不是这里的 Python 源码字符串，而是编译后的运行时字段值

    request: str = dspy.InputField()   # 输入：自然语言请求
    output: str = dspy.OutputField()   # 输出：期望是合法 JSON 字符串（不加 markdown 包裹）


class JsonSkill(dspy.Module):
    """JSON 输出技能模块，核心是一个 ChainOfThought predictor。"""

    def __init__(self):
        super().__init__()
        # ChainOfThought：让 LM 先写推理过程（reasoning），再给出最终输出（output）
        self.predictor = dspy.ChainOfThought(TaskSig)

    def forward(self, request: str):
        return self.predictor(request=request)


# 打印 baseline instruction（GEPA 优化前的初始状态）
# 访问路径：predictor.predict.signature.instructions
# 注意：ChainOfThought 内部多包了一层 Predict，所以要多一个 .predict
baseline_module = JsonSkill()
print(f"[BASELINE INSTRUCTION] {baseline_module.predictor.predict.signature.instructions}")

[BASELINE INSTRUCTION] Reply.


&emsp;&emsp;Cell 2 跑通后会打印 baseline instruction —— `"Reply."`（极弱的一个字母指令，刻意留给 `GEPA` 反思空间）。这就是 `GEPA` 即将优化的**编译产物字段** `signature.instructions`。注意 `predict.signature.instructions` 这条访问路径——`ChainOfThought` 在 DSPy 3.x 里内部包了 `Predict` 一层，所以要多一层 `.predict` 才能拿到真实指令字段。**关键概念**：`GEPA` 改的不是你 Python 源码里那行 `"""Reply."""` docstring 字面量，而是 DSPy 编译后内存里的 `signature.instructions` 字段（不动模型权重，也不写回源码）——这是 §3.3 问 2 那条动作链的物理落点。

&emsp;&emsp;**Cell 3：准备 5-10 个 JSON 输出训练样本**

&emsp;&emsp;`GEPA` 需要训练样本来评估 baseline 表现 + 找反思切入点。这里准备 6 条手工合成的样本——每条都是"自然语言请求 + 期望 JSON 输出"的对子。这些样本不需要复杂——只要有几条让 baseline 翻车（输出 markdown 代码块包裹），`GEPA` 就有反思素材。

In [17]:
# ── Cell 3：准备训练样本（6 条手工合成）─────────────────────────────
# 每条样本：request = 自然语言请求，output = 期望的合法 JSON 字符串
# 覆盖场景：简单对象、数组、空对象、布尔值、嵌套对象、数字数组
# baseline 在没有改进 instruction 的情况下，倾向于把 JSON 包在 Markdown JSON 围栏里——
# 这会导致 json.loads 解析失败，GEPA 反思后会针对这个问题改写 instruction

trainset = [
    dspy.Example(
        request="给我一个 user 对象，含 name+age",
        output='{"name":"Alice","age":30}'
    ).with_inputs("request"),
    dspy.Example(
        request="给我一个含 3 个商品的列表，每个有 id+price",
        output='[{"id":"A1","price":99.9},{"id":"A2","price":29.5},{"id":"A3","price":149}]'
    ).with_inputs("request"),
    dspy.Example(
        request="给我一个空对象",
        output='{}'
    ).with_inputs("request"),
    dspy.Example(
        request="给我一个含 boolean 的配置对象",
        output='{"enabled":true,"debug":false}'
    ).with_inputs("request"),
    dspy.Example(
        request="给我一个嵌套对象，user.profile.email",
        output='{"user":{"profile":{"email":"a@b.com"}}}'
    ).with_inputs("request"),
    dspy.Example(
        request="给我一个含数字数组的对象",
        output='{"scores":[80,90,100]}'
    ).with_inputs("request"),
]

print(f"[OK] {len(trainset)} 条训练样本就绪（覆盖 6 种 JSON 输出场景）")

[OK] 6 条训练样本就绪（覆盖 6 种 JSON 输出场景）


&emsp;&emsp;Cell 3 跑通后会看到 `[OK] 6 条训练样本就绪`。这 6 条样本里覆盖了 JSON 输出的几种典型场景——简单对象、数组、嵌套、布尔值、数字数组。baseline 在没看过 instruction 改进时倾向于把 JSON 包在 Markdown JSON 围栏里输出，导致 `json.loads` 解析失败——这正是 `GEPA` 反思后修补的方向。

&emsp;&emsp;**Cell 4：跑 GEPA（本讲最关键的一段代码）**

&emsp;&emsp;这是 Demo 3 的核心 cell——它做两件事：① 定义 `metric` 函数（判定一次输出是不是合法 JSON）；② 加载或生成 `gepa_cache.json`。课件已附带 cached 结果文件，你只需要跑加载分支即可，不依赖在线 API 也不收 token 费用。

In [18]:
# ============================================================
# Cell 4：metric 评分函数 + GEPA 优化器入口
# 对应真源码：tools/skill_provenance.py + dspy.GEPA
# ============================================================

# ── Part 1：定义 metric（评分标准）──────────────────────────
#    DSPy GEPA 强制要求 metric 函数必须有 5 个形参，
#    在 dspy.GEPA(...) 初始化时用 inspect.signature 校验，
#    少一个就直接抛 TypeError（BootstrapFewShot 只要 2-3 参，这是最大区别）
def metric(
    ex,                  # 训练样本（包含输入字段和期望输出）
    pred,                # 模型对该样本的实际输出（pred.output 是输出字段）
    trace=None,          # GEPA 执行轨迹（反思时用，metric 内部不需要使用）
    pred_name=None,      # 被评估的 predictor 名称（GEPA 内部传入）
    pred_trace=None,     # 该 predictor 的局部执行轨迹（GEPA 内部传入）
):
    """JSON 格式校验：pred.output 能被 json.loads 解析 → 1.0，否则 → 0.0。

    教学版只验 JSON 合法性，不验内容语义：
    - strip() 去除首尾空白，但 markdown 代码块（```json...```）不会被处理
    - 被 ``` 包裹的输出会让 json.loads 抛 JSONDecodeError → 返回 0.0
    - 这些 0.0 的样本就是 GEPA reflection_lm 的反思素材
    """
    try:
        json.loads(pred.output.strip())
        return 1.0   # 解析成功：合法 JSON
    except Exception:
        return 0.0   # 解析失败：格式错误（markdown 格式、多余文字等）


# ── Part 2：执行路径选择（cache 优先 / 真跑备选）─────────────
CACHE = Path("gepa_cache.json")  # 讲师课前真跑后落盘的结果文件

# 改为 True 可强制重跑 GEPA：会删除旧 cache，重新调 DeepSeek API
# 预计耗时：约 12 分钟 | 预计费用：约 ¥1-2 DeepSeek token 费
FORCE_REFRESH = True

if FORCE_REFRESH and CACHE.exists():
    CACHE.unlink()  # 删除旧 cache，触发下方 else 分支真跑
    print("[INFO] FORCE_REFRESH=True，已删除旧 cache，准备真跑 GEPA")

if CACHE.exists():
    # ── 主路径（默认）：直接加载 cache，零 API 调用、零等待 ──
    cached = json.loads(CACHE.read_text())
    print("[OK] Loaded cached GEPA result.")

else:
    # ── 真跑分支：需要配置 DEEPSEEK_API_KEY ──────────────────
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError(
            "请先按 §0.4 在项目根目录 .env 配置 DEEPSEEK_API_KEY 并重新加载环境再跑真跑分支"
        )

    print("[INFO] 开始真跑 dspy.GEPA（预计 12 分钟，预计 ¥1-2 DeepSeek token 费）...")

    # ── Part 3：初始化 GEPA 优化器 ──────────────────────────
    optimizer = dspy.GEPA(
        metric=metric,           # 评分函数：JSON 解析成功 → 1.0
        auto="light",            # 轻量预设（约 5 轮迭代）；"heavy" 轮数更多但耗时翻倍
        reflection_lm=dspy.LM(
            "deepseek/deepseek-chat"
            # ⚠️ 不用 deepseek-reasoner：reasoner 输出含 <think> token，
            #    会让 DSPy 解析器报错（issue #7489），因此反思 LLM 必须用 chat 模型
        ),
    )

    # compile() = 运行 GEPA 优化，把 JsonSkill 的 instruction 从 "Reply." 进化成精确规则段落
    # trainset 同时用作 训练集（反思素材）和 验证集（Pareto 评分）
    evolved = optimizer.compile(
        JsonSkill(),             # 待优化的模块（baseline instruction = "Reply."）
        trainset=trainset,       # 6 条训练样本（Cell 3 准备好的）
        valset=trainset,         # 验证集（教学版复用 trainset，生产版建议单独准备）
    )

    # ── Part 4：计算并对比准确率 ────────────────────────────
    baseline_module = JsonSkill()  # 重新创建未优化的原始模块作为对照组

    # 逐条跑 trainset，metric 打分后取平均值
    baseline_acc = sum(
        metric(ex, baseline_module(**ex.inputs())) for ex in trainset
    ) / len(trainset)

    evolved_acc = sum(
        metric(ex, evolved(**ex.inputs())) for ex in trainset
    ) / len(trainset)

    # ── Part 5：结果落盘到 gepa_cache.json ──────────────────
    cached = {
        "baseline_inst": baseline_module.predictor.predict.signature.instructions,  # 优化前 instruction 文本
        "evolved_inst":  evolved.predictor.predict.signature.instructions,           # 优化后 instruction 文本
        "baseline_acc":  baseline_acc,   # 通常 0.5（6 条中 3 条格式错误）
        "evolved_acc":   evolved_acc,    # 通常 1.0（6 条全部正确）
    }
    CACHE.write_text(json.dumps(cached, indent=2, ensure_ascii=False))

    print(f"[OK] GEPA 真跑完成，cache 已落盘 gepa_cache.json")
    print(f"     baseline_acc = {baseline_acc:.0%},  evolved_acc = {evolved_acc:.0%}")


2026/05/13 21:07:03 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 404 metric calls of the program. This amounts to 33.67 full evals on the train+val set.
2026/05/13 21:07:03 INFO dspy.teleprompt.gepa.gepa: Using 6 examples for tracking Pareto scores.


[INFO] FORCE_REFRESH=True，已删除旧 cache，准备真跑 GEPA
[INFO] 开始真跑 dspy.GEPA（预计 12 分钟，预计 ¥1-2 DeepSeek token 费）...


GEPA Optimization:   0%|          | 0/404 [00:00<?, ?rollouts/s]2026/05/13 21:07:04 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 6 (16.7%)
2026/05/13 21:07:04 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.16666666666666666 over 6 / 6 examples
GEPA Optimization:   1%|▏         | 6/404 [00:00<00:19, 20.54rollouts/s]2026/05/13 21:07:04 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.16666666666666666


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 179.44it/s]

2026/05/13 21:07:04 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)
2026/05/13 21:07:04 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predictor.predict: You are an AI assistant that generates data structures based on user requests. When given a request, you must:

1. First, in a "reasoning" section, think step by step about what the user is asking for. Identify the exact structure, fields, data types, and nesting required. If the request is ambiguous, choose a reasonable default (e.g., placeholder strings for emails, numeric IDs, realistic prices).

2. Then, in an "output" section, produce the response in one of two formats, depending on the request:
   - If the request asks for a JSON-like structure (e.g., nested objects, arrays, key-value pairs), output it as a valid JSON block using triple backticks with `json`.
   - If the request asks for a list, table, or plain enumeration (e.g., "a list of 3 items with id and price"), output it as a clear, human-

...


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 65.44it/s]

2026/05/13 21:07:18 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/13 21:07:18 INFO dspy.teleprompt.gepa.gepa: Iteration 125: All subsample scores perfect. Skipping.
2026/05/13 21:07:18 INFO dspy.teleprompt.gepa.gepa: Iteration 125: Reflective mutation did not propose a new candidate
2026/05/13 21:07:18 INFO dspy.teleprompt.gepa.gepa: Iteration 126: Selected program 2 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 67.54it/s]

2026/05/13 21:07:18 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/13 21:07:18 INFO dspy.teleprompt.gepa.gepa: Iteration 126: All subsample scores perfect. Skipping.
2026/05/13 21:07:18 INFO dspy.teleprompt.gepa.gepa: Iteration 126: Reflective mutation did not propose a new candidate
GEPA Optimization: 100%|█████████▉| 402/404 [00:14<00:00, 27.84rollouts/s]

[OK] GEPA 真跑完成，cache 已落盘 gepa_cache.json
     baseline_acc = 17%,  evolved_acc = 100%


&emsp;&emsp;Cell 4 跑通后会看到 `[OK] Loaded cached GEPA result.`——这是 `CACHE.exists()` 主路径，从 `gepa_cache.json` 直接加载讲师课前真跑的结果，零成本零等待。如果你想自己亲手跑一次（譬如改了 `trainset` 换成自己业务任务），把 `FORCE_REFRESH = False` 改成 `True` 即可——代码会删除旧 cache、调 DeepSeek API 跑 `dspy.GEPA` 约 12 分钟（预计 ¥1-2 token 费，网络波动会有少量上下浮动），跑完落盘新 cache 供 Cell 5 加载。

&emsp;&emsp;关于 `gepa_cache.json` 的三件事你最好心里有底：**第一，cached 数据来源**——讲师课前用 `dspy.GEPA(metric=metric, auto="light", reflection_lm=dspy.LM("deepseek/deepseek-chat"))` 在同一份 `trainset` 上真跑一次后落盘的，不是手工编造的"教学示例"。**第二，真实性**——cached 内容跟你本地真跑等价（同一 trainset / 同一 metric / 同一 baseline LM），只是省下了那 12 分钟和 ¥1-2 token 费。**第三，复跑入口**——`FORCE_REFRESH` 标志位是你想强制重跑时的唯一开关，改成 `True` 后 Cell 4 会删 cache → 调 API → 真跑 → 落新 cache，全自动；不改的话默认零成本演示，不会消耗任何 token。

&emsp;&emsp;**Cell 5：看 baseline → evolved 的 instructions diff（高潮）**

&emsp;&emsp;最后一个 cell 把 baseline / evolved 两个 instruction 字符串、加 score 数字一起打印出来。这是全课最强的"看见"时刻——**亲眼看到 `GEPA` 把一句极弱指令变成一段多约束英文规则文本**，加上具体约束（"raw JSON / no markdown / no backticks"）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513144256998.png" width=60%></div>

In [19]:
# ── Cell 5：打印 baseline → evolved 的 instruction 对比 + 准确率提升 ──
# 这是 Demo 3 的"高潮"——亲眼看到 GEPA 把极弱 instruction 进化成精确规则

SEP = "=" * 60

print(SEP)
print("=== BASELINE INSTRUCTION（GEPA 优化前）===")
print(SEP)
print(cached["baseline_inst"])          # 极弱 baseline：只有 "Reply." 一个字

print()
print(SEP)
print("=== EVOLVED INSTRUCTION（GEPA 反思后）===")
print(SEP)
print(cached["evolved_inst"])           # 进化后：一段多约束的英文规则段落

print()
print(SEP)
print(f"Baseline accuracy : {cached['baseline_acc']:.0%}")
print(f"Evolved  accuracy : {cached['evolved_acc']:.0%}")
print(f"绝对提升          : {(cached['evolved_acc'] - cached['baseline_acc']):.0%}")
print(SEP)

=== BASELINE INSTRUCTION（GEPA 优化前）===
Reply.

=== EVOLVED INSTRUCTION（GEPA 反思后）===
When the user asks you to create an object, you must always output a valid JSON object without any surrounding backticks, code block formatting, or any other text. The output should be the raw JSON object itself, using only double quotes for property names and string values — never single quotes (Python dict style) or any non-JSON syntax.

Follow these rules:
1. Never wrap the JSON in ```json ... ``` or any other code block markers.
2. Do not include any reasoning, explanations, or additional text before or after the JSON.
3. Use only double quotes `"` for all property names and string values. Never use single quotes.
4. If the user does not specify property names or values, invent plausible, realistic example values (such as common placeholder names, typical ages, common boolean patterns) to make the object complete and realistic.
5. If the user explicitly mentions a desired structure (e.g., user with n

&emsp;&emsp;Cell 5 跑通后输出大致长这样（cached 结果一次锁定，可重复观察）（注：这段 evolved instruction 是本次 trainset / metric 下 `GEPA` 进化出的**一个候选结果**，不是 `GEPA` 固定模板——换你自己的业务 trainset，进化出的 instruction 会不同。）：

&emsp;&emsp;> **【关键时刻话术】**：看，`GEPA` 改的就是 `predictor.predict.signature.instructions` 这个字段——一个字 `"Reply."` 变成了一段精确的英文规则段落，加了具体约束（"reasoning + output 双字段" / "use double quotes for all strings" / "never wrap in markdown code block"）。这就是 §3.3 那个抽象的"反思变异"动作链在你眼前真实跑出来的形态——`GEPA` 反思 LLM（`deepseek-chat`）读了 baseline 在 6 个样本上的失败案例（譬如把商品列表输出成 markdown 编号格式），自己写出一段精确英文规则段落，accuracy 从 50% 拉到 100%。**整个过程模型权重没变，只换了一句 prompt**。

<p align="center">表 3-2 Demo 3 能展示的 vs 看不到的（诚实告知，避免学员误解）</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 能让你看见 | 看不到（需要更深的工具或 log） |
| :---: | :---: |
| Baseline vs Evolved instruction 文本 diff | Pareto 前沿动态选择过程（`GEPA` 内部黑盒） |
| Baseline vs Evolved score 真实提升 | 反思 NL 诊断原文（需 `log_dir=` 参数后再读文件） |
| `DSPy` 控制台进度条 + 候选评分变化 | Merge 机制（两候选合并）等高阶细节 |
| "`GEPA` 改 `instructions` 不改 `skill_text`" 这个核心机制 | 多目标评估（本 demo 仅用单一 metric=accuracy） |

&emsp;&emsp;表 3-2 让我们知道 30 行 demo 的能力边界——它能让你"看见 + 对比"，足够支撑本课的核心教学目标（建立"`GEPA` 改 `instructions` 不改 `skill_text`"的"心智模型"（mental model，在脑中建立的理解框架））；但 `Pareto` 前沿、merge 机制、多目标评估这些 `GEPA` 内部细节属于研究级深度，超出本课范围。生产建议和 `MIPROv2` 兜底路径见下一节 §3.5。

### 3.5 GEPA 真实作用 + 实战建议

&emsp;&emsp;Demo 3 已经让我们亲眼看见 `GEPA` 干了什么——它改的是 `predictor.predict.signature.instructions`（DSPy 3.x 中 `ChainOfThought` 内部 `Predict` 的指令字段访问路径），一个字 `"Reply."` 进化成一段精确的英文规则段落（覆盖 reasoning+output 双字段 / 双引号要求 / markdown 禁止 / 嵌套对象规范 等约束），accuracy 从 50% 升到 100%。**变的是 instruction 文本，不是模型权重**——这就是 §3.2 表 3-1 左列"机制层进化"的物理载体。

&emsp;&emsp;**真实数字对照**：Demo 3 在 JSON 输出任务上是 `50%→100%`（讲师 2026-05 在 `harness` 环境用 `deepseek-chat` 实测）；GEPA 论文（ICLR 2026 Oral）在 `MATH benchmark` 上是 `70%→92%`。两组数字的任务难度完全不同，不能互比，但方向一致——反思变异能显著提升 accuracy。记住数字的同时记住一件事：它们都是"特定任务 + 特定 metric + 特定 trainset"下的结果，换成你自己的业务任务，结果会不同。

&emsp;&emsp;**实战建议 · 渐进路径**：想做 prompt 自动优化，推荐先从 `MIPROv2` 跑通基线，再考虑切 `GEPA`。`MIPROv2` 是 DSPy 内置的另一款优化器（走贝叶斯搜索路线，文档全、案例多、跑了 2 年），切换方法很简单——把 §3.4 Cell 4 那行 `dspy.GEPA(...)` 换成 `dspy.MIPROv2(metric=metric, auto="light")` 即可。

&emsp;&emsp;**另一条独立路径**——如果你不想绑 DSPy 框架，直接 `pip install gepa`（独立包 `v0.1.1`，底层走 `litellm`——跨模型统一调用库，不依赖 dspy）也能跑同套 GEPA 算法。两者关系是"独立 pip 包提供算法 + DSPy 包做 `dspy.Module` 集成"——脱离 DSPy 用就装独立包，想跟 DSPy 既有的 `Module / Signature` 体系无缝集成就用 `dspy.GEPA`。

&emsp;&emsp;**一句话闭环**：以后面对任何 agent 产品的卖点，套同一套四步法——**看话术 → 看源码 → 亲手跑 demo → 写进选型笔记**。这才是 30 行 GEPA 真跑给你的真正价值：不是那个数字，而是这套方法论。

&emsp;&emsp;> **【学完本章你已经掌握】**：① 自进化飞轮的真实边界——进化的是机制层（`skill` / `MEMORY` / tool 模式），不是模型层（参数 / 推理 / 知识边界）；② `GEPA` 是什么——`Genetic-Pareto` 反思变异 prompt 优化器，核心动作是"让大模型读自己上次错在哪 → 自动改写 instruction"；③ 30 行 `dspy.GEPA` 真跑——亲眼看到 `instructions` 一个字变一段英文规则段落、`50% → 100%` 提升；④ 实战渐进路径——先用 `MIPROv2` 跑通基线再切 `GEPA`，或独立装 `gepa` pip 包脱离 DSPy 用。

## <center>第4章：系列交付</center>

&emsp;&emsp;到这里 `Harness Engineering` 系列五讲全部讲完。最后一章不教新内容，只做三件事——先用第四节课第 3 章 抽出的**四维评价尺**把四家 harness 在同一坐标系上摆清（§4.1），再把今天本课讲过的七件具体产物清点交付（§4.2），最后给整个系列做一次五讲交付汇总（§4.3）。一节不长，但希望你认真读完——这是你回去之后再次面对真实选型 / 真实 agent 产品时的回血手册。

### 4.1 四家四维横评 + 选型决策地图

&emsp;&emsp;到这里本课的具体内容已经讲完，但 `Harness Engineering` 这个系列还差最后一个动作——把"会评价 `Hermes` 一家"升级为"能选型任何 harness"。这一节我们用 第四节课第 3 章 抽出的**四维评价尺**（`GC 动态化 / AC 架构约束 / CE 上下文工程 / 入口治理`，前 3 维直接复用第一节课三支柱，第 4 维是入口治理补充维度），把 4 家 harness 放上同一坐标系，让我们拿到一把可复用于任何选型场景的尺。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260513144256975.png" width=60%></div>

&emsp;&emsp;**OpenClaw 简介**：另一款开源 agent harness，TypeScript 栈，定位 Personal AI Assistant 偏消费级。**前四讲未出现，今天终章作为第四参照引入**——它跟 `Anthropic Claude Code` / `OpenAI Codex` / `LangChain DeepAgents` 三家在"研发场景"对齐，但选择了完全不同的"消费场景"路径，刚好提供一个非研发视角的对照锚点。**最显著特征**是 `Skill Marketplace`（skill 社区市场，可分发/订阅他人写好的 skill）+ `Live Canvas`（多渠道实时画布）。工程姿态对照 `Hermes` 是"`Skill` 社区市场 + 多渠道 + Live Canvas" vs "`Skill` 自学习 + `Curator` + 19 平台"——前者把 skill 当**社区资产分发**，后者把 skill 当 **agent 自己写出来的私产管理**。两条路径都用 skill，但社区/私产的分野决定了它们的工程取舍完全不同。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-1 四家 Harness 在四维评价尺上的位置（系列终章）</font></p>
<div class="center">

| 维度 / 系统 | Anthropic Claude Code | OpenAI Codex | LangChain DeepAgents | OpenClaw |
|---|---|---|---|---|
| **GC 动态化**（skill/记忆是否后台自动整理） | 弱（skill 用户手写）| 弱 | 居中（用户决定）| 偏静态 |
| **AC 架构约束**（沙箱/权限/边界强度） | **强约束**（kernel-space sandbox + worktree subagent）| 中等 | 居中 | 中等（产品体验约束）|
| **CE 上下文工程**（上下文管理工程化程度） | 强（KV-cache + 自动 compact）| 中等（OpenAI infra 内置）| 强（state schema + middleware）| 中等 |
| **入口治理**（单/多平台锁定） | 集中（单 IDE 入口）| **强集中**（紧密集成）| 居中 | **强分治**（产品体验路径）|
| **代表场景** | 编码项目（精度优先）| AI Pair Programming | 框架自由组合 | 产品级 AI 助手 |
| **典型代价** | 用户持续维护 SKILL.md 成本 | 单平台锁定 | 配置复杂度 | 学习曲线 |
| **社区典型用法** | 独立或 + Hermes 后台 | 单独使用 | 框架级二次开发 | 开箱即用 |

</div>


&emsp;&emsp;**Hermes 自家在这把尺上的参考位置**（不进对照表，单独说明）——`Hermes` 在 GC 动态化是**强动态**（`Nudge + Curator + ContextVar` 三件套是本系列重点讲的取舍），AC 架构约束**弱**（信任 LLM 自由调用），CE 上下文工程**强**（三层记忆 + Frozen Snapshot），入口治理**分治弱**（双入口未统一）。把 Hermes 自家放在这把尺上我们会发现——它跟其他四家不在同一象限，是"另一种取舍"，不是任何一家的升级版。

&emsp;&emsp;**讲解节奏（15 min 拆分）**：

&emsp;&emsp;**第一段（5 min）四维各家点位**——`Anthropic Claude Code` 在"AC 架构约束"轴上是**强约束**，这是它最显著的差异化取舍——`kernel-space sandbox`（内核级沙箱）+ `worktree subagent`（独立 git 工作树子代理，每个子任务在隔离副本里跑）让你给它写权限不必担心它把项目仓库搞坏。`OpenAI Codex` 在"入口治理"上是**强集中**——它跟 OpenAI 自家产品（ChatGPT / Codex CLI）紧密集成，单平台锁定换来的是开箱即用的极致体验。`LangChain DeepAgents` 四维都偏中——它的定位是"框架"不是"产品"，所有维度都让用户决定，代价是配置复杂度。`OpenClaw` 在"入口治理"上偏分治——它走的是产品体验路径，把 Agent 拆成多个独立模块让用户组合。

&emsp;&emsp;**第二段（5 min）对照"代表场景 + 典型代价"**——这一段不是让你选哪家好哪家坏，而是让我们看到**没有最好，只有最适合**。如果你做的是高精度编码项目（譬如重构核心服务、多文件批量改造），`Claude Code` 的强约束是你的护甲——它不会乱写文件。如果你做 AI Pair Programming（结对编程，编辑器内随手喊一声 "fix this"），`Codex` 的强集中给你的是延迟低、体验顺滑。如果你要二次开发自己的 agent 框架，`DeepAgents` 让你拼装。如果你要个开箱即用的 AI 助手，`OpenClaw` 帮你最快上手。**每家都用"放弃某些维度"换取"极致某些维度"**。

&emsp;&emsp;**第三段（5 min）选型决策地图**——你课后看自己项目在四维上分别需要哪一端：GC 动态化要不要后台自动整理？AC 架构约束需要多严？CE 上下文工程要不要框架内置？入口治理要单平台锁定还是组合自由？看你的项目落在哪家附近——这就是你的选型决策。学完这个对照表，给自己的项目选 harness 时不再问"`XX` 家好不好"，而是问"`XX` 家的位置跟我项目需要的位置匹配吗"。

&emsp;&emsp;> **【贯穿全课的核心论断】**：`Hermes` 不是 `Claude Code` 升级版，是另一种取舍。**四维评价尺**让我们看到的是——四家在四维上的位置完全不一样，每家都用"放弃某些维度"换取"极致某些维度"。学完这个对照表，给自己的项目选 harness 时不再问"哪家最强"，而是问"哪家的位置跟我项目需要的位置匹配"。这才是这堂课要给你的尺。

### 4.2 学员出讲带走：今天的七件产物

&emsp;&emsp;今天本课这一讲下来，你应该带走七件具体产物——每一件都对应今天讲的某一段、且都是可复用资产，不是一次性消耗品：

<p align="center">表 4-2 本课学员出讲带走清单（七件产物）</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| # | 产物 | 数量 | 用途 |
| :---: | :---: | :---: | :---: |
| ① | `Nudge × Curator` 自进化飞轮图（§3.1） | 1 张 | 理解整个自进化闭环 + 真实进化边界 |
| ② | `Curator` 机制源码 walk 笔记（第 1 章） | 1 份 | 课后复习双 Phase 流程 + 五条护栏 + idle 触发 |
| ③ | `curator dry-run` 演示记录（Demo 1） | 1 份 | 朋友圈 / 简历素材，证明你跑过生产级源码 |
| ④ | `skill` 血统隔离机制笔记（第 2 章 + Demo 2） | 1 份 | 理解 `ContextVar` 工程取舍 + 5 月修补两个 commit |
| ⑤ | `GEPA` 三问概念卡（§3.3） | 1 份 | 30 秒回顾 GEPA 是什么 / 怎么做 / 为什么有效 |
| ⑥ | `30` 行 `dspy.GEPA` 真跑 cached 输出（Demo 3） | 1 份 | 课后改 task 复跑；做 prompt 优化项目时直接 import 模板 |
| ⑦ | `GEPA` 实战渐进路径笔记（§3.5） | 1 份 | 想做 prompt 自动优化时直接拿来用：MIPROv2 → GEPA → 独立 pip 包三段台阶 |

&emsp;&emsp;表 4-2 的每一件你都应该在课后 24 小时内整理一遍——把课堂笔记、demo 跑通的截图、自测命令的输出全部归档到一个目录。这些产物的价值不在于"今天用了"，而在于"以后你每次面对一个新 agent 产品时它们都能复用"。第六项尤其值得花时间——`gepa_cache.json` 这个文件你课后可以改 `trainset` 重跑，复用整套 `dspy.GEPA` 框架做你自己业务里的 prompt 优化项目。

### 4.3 系列收束：五讲完整交付清单

&emsp;&emsp;今天是 `Harness Engineering` 系列的**终章**。回头看这五讲走过的路径——第一节课给认知地图、第二节课让你手搓、第三节课用框架装、第四节课读生产源码、本课看清边界 + 横评收束——是一条**从理论词汇到工程取舍 walk 的完整链路**。下面这张表把五讲的核心交付汇总：

<p align="center">表 4-3 Harness Engineering 五讲完整交付映射</p>
<div class="center">

<style>
/* 强制表格居中、自动换行并适应单元格宽度 */
.rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: auto !important;
    margin-right: auto !important;
    width: auto !important; /* 允许表格根据内容收缩 */
    max-width: 100%; /* 防止表格溢出单元格 */
    table-layout: fixed; /* 固定布局算法，对长文本换行至关重要 */
}
.rendered_html th, .jp-RenderedHTMLCommon th,
.rendered_html td, .jp-RenderedHTMLCommon td {
    white-space: normal !important; /* 允许自动换行 */
    word-wrap: break-word; /* 对长单词或URL进行强制换行 */
    text-align: left; /* 默认内容左对齐 */
}
.rendered_html th, .jp-RenderedHTMLCommon th {
    text-align: center !important; /* 表头文本居中 */
}
</style>

| 讲次 | 核心定位 | 你拿到的核心资产 |
| :---: | :---: | :---: |
| L1 认知地图 | 三支柱 + 八大故障的诊断清单 | 给手头任何 agent 项目命名能力的"概念尺" |
| L2 mini Harness 手搓 | 11 机制亲手实现 + Baseline 1104 → Full 15053 tokens | 知道"骨架长什么样"的物理"心智模型" |
| L3 DeepAgents 实战 | 4 大组件 + 8 步流水线 + `deepagents==0.5.3` | 用框架装机制的"生产姿势" |
| L4 Hermes 记忆 + Nudge | 四维评价尺 + 三层记忆架构 + Nudge 计数器机制 | 评价 harness 的“工程尺” |
| L5 Curator + 诚实边界 | Curator 双 Phase + ContextVar 血统 + 批判性思维四步法 + 四家四维横评 | **选型 + 识破任何 marketing 的最后一公里** |

&emsp;&emsp;> **【系列终章四件带走】**：5 讲下来你应该带走四件东西——

① 一把**四维评价尺**（`GC 动态化 / AC 架构约束 / CE 上下文工程 / 入口治理`）——评价**任何** harness；

② 一份**继承自第四节课**的 `Hermes` 默认架构图（热层 `MEMORY/USER` + 冷层 `SQLite/FTS5/摘要` + 扩展层 `Memory Provider` 插件——后者默认全部关闭，需用户 `config.yaml` 显式启用）——看清营销话术 vs 源码真相；

③ 一张**四家四维横评对照表 + 你自己项目的四维位置坐标**——给自己的项目选 harness 时不被任何一家的能力清单或 stars 数淹没；

④ **亲手跑过 `GEPA` 真版本看见 `instructions` 进化** + 拿到 `MIPROv2 → GEPA → 独立 pip 包` 三段实战路径——以后做任何 prompt 自动优化项目都有一条可以直接 import 的起点模板。

&emsp;&emsp;`Harness Engineering` 系列五讲到此正式结束——是从“听说过 agent harness”到“能评价、能选型、能识破”完整知识链的闭合。下一次你打开任何 agent 产品的 README 或 PDF，希望心里那把**四维尺**自动启动、那个“亲手 grep 验证”的本能立刻冒出来——这就是这五讲想给你的最大的礼物。这条主线对未来的新产品 / 新论文保持开放，欢迎你课后跑通选型决策或踩到新坑后给反馈。